# Texto y lenguaje

Tokenización y sílabas, agentes de RL sobre texto, BERT, difusión de texto.

Parte del **laboratorio neuronal**. Cada sección es un experimento independiente: ejecuta solo la que te interese, de arriba hacia abajo (cada una instala lo que necesita).

## Agente PPO que predice la siguiente palabra

Antes se llamaba «Agente PPO para segmentar texto en español», pero lo que hace es predecir la palabra que sigue.

In [ ]:
# Nota: pese al título, este agente predice la siguiente palabra (no segmenta). Medido en el banco:
# laboratorio/banco/siguiente_palabra.py (resultados en laboratorio/banco/resultados-01-generativos.md).
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- NUEVO: Define las oraciones globalmente o pásalas a la clase ---
# Opción 1: Definir las oraciones antes de la clase para usarlas en el vocabulario
sentences_for_env = [
    "la complejidad de la inteligencia artificial moderna radica en la intrincada interacción de algoritmos de aprendizaje profundo y vastos conjuntos de datos no estructurados",
    "el descubrimiento de nuevas galaxias distantes por medio de telescopios espaciales avanzados ha proporcionado a los astrofísicos una perspectiva sin precedentes sobre la expansión acelerada del universo",
    "la implementación de soluciones de energía renovable a gran escala presenta desafíos logísticos significativos que requieren una colaboración internacional y una inversión sustancial en infraestructura",
    "el estudio de los patrones migratorios de las aves marinas revela conexiones sorprendentes entre los ecosistemas oceánicos globales y la salud de las poblaciones de peces en aguas profundas",
    "la evolución de las redes neuronales convolucionales ha revolucionado el campo del reconocimiento de imágenes permitiendo la detección y clasificación precisa de objetos en entornos complejos y dinámicos",
    "la investigación sobre la genética de enfermedades autoinmunes ha desvelado vías moleculares intrincadas que podrían conducir al desarrollo de terapias personalizadas dirigidas a mecanismos inmunológicos específicos",
    "la exploración de los océanos profundos ha revelado una biodiversidad asombrosa y ecosistemas únicos que dependen de fuentes de energía quimiosintéticas distintas de la fotosíntesis solar",
    "el análisis comparativo de lenguajes de programación modernos demuestra la creciente tendencia hacia paradigmas declarativos y la abstracción de detalles de bajo nivel para mejorar la productividad del desarrollador",
    "la influencia del cambio climático en los patrones meteorológicos extremos ha generado una necesidad urgente de estrategias de adaptación y mitigación a nivel global para proteger las comunidades vulnerables",
    "el desarrollo de nuevos materiales con propiedades cuánticas ofrece perspectivas prometedoras para la creación de dispositivos electrónicos ultraeficientes y computadoras con capacidades sin precedentes"
]

class SpanishTextEnv:
    def __init__(self, vocabulary, sentences): # Añade 'sentences' como parámetro
        self.vocabulary = vocabulary
        self.word_to_idx = {word: i for i, word in enumerate(vocabulary)}
        self.idx_to_word = {i: word for i, word in enumerate(vocabulary)}
        self.action_space_size = len(vocabulary)
        self.current_sentence_idx = 0
        self.sentences = sentences # Usa las oraciones pasadas como parámetro
        self.reset()

    def reset(self):
        self.current_sentence_idx = np.random.randint(len(self.sentences))
        self.current_words = self.sentences[self.current_sentence_idx].split()
        self.current_word_index = 0
        self.done = False

        initial_word_idx = self.word_to_idx.get(self.current_words[self.current_word_index], 0)
        state = self._one_hot_encode(initial_word_idx)
        return state

    def step(self, action_idx):
        predicted_word = self.idx_to_word[action_idx]

        self.current_word_index += 1
        reward = 0.0
        info = {}

        if self.current_word_index < len(self.current_words):
            expected_word = self.current_words[self.current_word_index]

            if predicted_word == expected_word:
                reward = 1.0
            elif predicted_word in self.vocabulary:
                reward = -0.1
            else:
                reward = -0.5

            next_state_idx = self.word_to_idx.get(expected_word, 0)
            next_state = self._one_hot_encode(next_state_idx)
            self.done = False

        else:
            expected_last_word = self.current_words[-1]
            if predicted_word == expected_last_word:
                reward = 2.0
            else:
                reward = -1.0

            next_state = self._one_hot_encode(self.word_to_idx.get("<EOS>", 0))
            self.done = True

        return next_state, reward, self.done, info

    def _one_hot_encode(self, idx):
        vec = torch.zeros(self.action_space_size, dtype=torch.float32)
        if 0 <= idx < self.action_space_size:
            vec[idx] = 1.0
        return vec.unsqueeze(0)

# Define the PPOAgent class (sin cambios)
class ActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim, action_dim):
        super(ActorCritic, self).__init__()

        # actor
        self.actor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1)
        )

        # critic
        self.critic = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, state):
        return self.actor(state), self.critic(state)

class RolloutBuffer:
    def __init__(self):
        self.actions = []
        self.states = []
        self.log_probs = []
        self.rewards = []
        self.dones = []

    def clear(self):
        del self.actions[:]
        del self.states[:]
        del self.log_probs[:]
        del self.rewards[:]
        del self.dones[:]

class PPOAgent:
    def __init__(self, input_dim, hidden_dim, action_dim, lr, gamma, K_epochs, eps_clip):
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs

        self.buffer = RolloutBuffer()

        self.actor_critic = ActorCritic(input_dim, hidden_dim, action_dim)
        self.optimizer = torch.optim.Adam([
            {'params': self.actor_critic.actor.parameters(), 'lr': lr},
            {'params': self.actor_critic.critic.parameters(), 'lr': lr}
        ])

        self.MseLoss = nn.MSELoss()

    def select_action(self, state):
        with torch.no_grad():
            action_probs, _ = self.actor_critic(state)
            dist = Categorical(action_probs)
            action = dist.sample()
            log_prob = dist.log_prob(action)
        return action, log_prob

    def get_action(self, state):
        with torch.no_grad():
            action_probs, _ = self.actor_critic(state)
            dist = Categorical(action_probs)
            action = dist.sample()
        return action.item()

    def update(self):
        rewards = []
        discounted_reward = 0
        for reward, is_done in zip(reversed(self.buffer.rewards), reversed(self.buffer.dones)):
            if is_done:
                discounted_reward = 0
            discounted_reward = reward + (self.gamma * discounted_reward)
            rewards.insert(0, discounted_reward)

        rewards = torch.tensor(rewards, dtype=torch.float32).to(device)
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-7)

        old_states = torch.cat(self.buffer.states, dim=0).detach().to(device)
        old_actions = torch.tensor(self.buffer.actions, dtype=torch.long).to(device)
        old_log_probs = torch.tensor(self.buffer.log_probs, dtype=torch.float32).to(device)

        for _ in range(self.K_epochs):
            log_probs, state_values = self.actor_critic(old_states)
            dist = Categorical(log_probs)
            log_probs = dist.log_prob(old_actions)
            state_values = torch.squeeze(state_values)

            advantages = rewards - state_values.detach()

            ratios = torch.exp(log_probs - old_log_probs.detach())

            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages

            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss = self.MseLoss(state_values, rewards)

            loss = policy_loss + 0.5 * value_loss

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        self.buffer.clear()

    def save(self, checkpoint_path):
        torch.save(self.actor_critic.state_dict(), checkpoint_path)

    def load(self, checkpoint_path):
        self.actor_critic.load_state_dict(torch.load(checkpoint_path, map_location=lambda storage, loc: storage, weights_only=True))

    def to(self, device):
        self.actor_critic.to(device)
        return self

# --- Configuración y Bucle Principal Actualizado ---
config = {
    "lr": 1e-4,
    "gamma": 0.99,
    "K_epochs": 4,
    "eps_clip": 0.2,
    "num_episodes": 100000,
    "model_path": "/content/ppo_spanish_text_model.pt",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- CORRECCIÓN: Construir vocabulario a partir de 'sentences_for_env' ---
vocabulary = list(set(" ".join(sentences_for_env).split())) + ["<EOS>"]

env = SpanishTextEnv(vocabulary=vocabulary, sentences=sentences_for_env) # Pasa las oraciones al entorno

agent = PPOAgent(
    input_dim=env.action_space_size,
    hidden_dim=256,
    action_dim=env.action_space_size,
    lr=config["lr"],
    gamma=config["gamma"],
    K_epochs=config["K_epochs"],
    eps_clip=config["eps_clip"]
).to(device)

reward_history = []
best_score = -float('inf')

for episode in tqdm(range(config["num_episodes"])):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        state_tensor = state.to(device)
        action, log_prob = agent.select_action(state_tensor)
        next_state, reward, done, _ = env.step(action.item())

        agent.buffer.states.append(state_tensor.cpu())
        agent.buffer.actions.append(action.cpu())
        agent.buffer.log_probs.append(log_prob.cpu())
        agent.buffer.rewards.append(reward)
        agent.buffer.dones.append(done)

        state = next_state
        total_reward += reward

    agent.update()
    reward_history.append(total_reward)

    if total_reward > best_score:
        best_score = total_reward
        agent.save(config["model_path"])

print(f"Mejor puntaje obtenido: {best_score:.2f}")

# --- Nueva función para generar texto ---
def generate_text_sequence(agent, env, start_word=None, max_length=10, device='cpu'):
    generated_sequence = []

    if start_word is None:
        # Si no se especifica una palabra de inicio, toma la primera palabra de una oración aleatoria
        # Asegúrate de que las oraciones estén accesibles para la generación
        initial_sentence = env.sentences[np.random.randint(len(env.sentences))].split()
        current_word = initial_sentence[0]
    else:
        current_word = start_word

    generated_sequence.append(current_word)

    current_word_idx = env.word_to_idx.get(current_word, env.word_to_idx.get("<EOS>", 0))
    state = env._one_hot_encode(current_word_idx).to(device)

    for _ in range(max_length - 1):
        action_idx = agent.get_action(state)

        predicted_word = env.idx_to_word[action_idx]
        generated_sequence.append(predicted_word)

        if predicted_word == "<EOS>":
            break

        next_word_idx = env.word_to_idx.get(predicted_word, env.word_to_idx.get("<EOS>", 0))
        state = env._one_hot_encode(next_word_idx).to(device)

    return " ".join(generated_sequence)

# --- Después del bucle de entrenamiento, genera y muestra secuencias ---
print("\n--- Secuencias de texto generadas después del entrenamiento ---")
agent.load(config["model_path"])
agent.to(device)

num_sequences_to_generate = 5
for i in range(num_sequences_to_generate):
    generated_text = generate_text_sequence(agent, env, start_word=None, max_length=5, device=device)
    print(f"Secuencia {i+1}: {generated_text}")

# --- Visualización de resultados (opcional) ---
plt.figure(figsize=(10, 6))
plt.plot(reward_history)
plt.xlabel("Episodios")
plt.ylabel("Reward acumulado")
plt.title("Entrenamiento PPO para Predicción de Siguiente Palabra en Español")
plt.grid(True)
plt.show()

## Segmentador de sílabas

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# === Dataset pequeño manual ===
data = [
    ("manzana", [0, 1, 0, 1, 0, 0]),
    ("pera", [0, 1, 0, 0]),
    ("azúcar", [0, 1, 0, 1, 0, 0]),
    ("año", [0, 1, 0]),
    ("gracias", [0, 1, 0, 0, 1, 0, 0]),
    ("murciélago", [0, 1, 0, 0, 1, 0, 1, 0, 0, 0]),
    ("dulce", [0, 1, 0, 0, 0]),
    ("frutas", [0, 1, 0, 1, 0, 0]),
    ("flores", [0, 1, 0, 0, 0, 0]),
    ("legumbres", [0, 1, 0, 1, 0, 0, 1, 0, 0]),
]

# === Vocabulario por caracteres ===
alphabet = sorted(set("".join(word for word, _ in data)))
char2idx = {c: i + 1 for i, c in enumerate(alphabet)}  # 0 = padding
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(char2idx) + 1

# === Dataset personalizado ===
class SyllableDataset(Dataset):
    def __init__(self, data, max_len=12):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def encode_word(self, word):
        x = [char2idx[c] for c in word]
        x += [0] * (self.max_len - len(x))
        return torch.tensor(x)

    def encode_labels(self, labels):
        y = labels + [0] * (self.max_len - len(labels))
        return torch.tensor(y)

    def __getitem__(self, idx):
        word, labels = self.data[idx]
        return self.encode_word(word), self.encode_labels(labels)

# === Modelo básico ===
class SyllableSegmenter(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.linear = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.linear(x).squeeze(-1)
        return x

# === Entrenamiento ===
def train(model, dataloader, epochs=100, lr=0.01):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in dataloader:
            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d}, LR: {scheduler.get_last_lr()[0]:.5f}, Loss: {total_loss:.4f}")

# === Inferencia simple ===
def predict(model, word, threshold=0.5, max_len=12):
    model.eval()
    x = torch.tensor([[char2idx.get(c, 0) for c in word] + [0]*(max_len - len(word))])
    with torch.no_grad():
        logits = model(x)
        probs = torch.sigmoid(logits).squeeze(0)
        cuts = [i for i, p in enumerate(probs[:len(word)]) if p > threshold]
        result = ""
        for i, c in enumerate(word):
            result += c
            if i in cuts:
                result += "|"
        return result

# === Configuración ===
dataset = SyllableDataset(data)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
model = SyllableSegmenter(vocab_size)

# === Entrenar ===
train(model, dataloader, epochs=1000, lr=0.001)

# === Probar ===
test_words = ["murciélago", "gracias", "flores", "frutas", "legumbres", "dulce"]
for w in test_words:
    print(f"{w:12s} → {predict(model, w)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Vocabulario fijo
VOCAB = list("abcdefghijklmnñopqrstuvwxyzáéíóúü") + ["<pad>"]
char2idx = {c: i for i, c in enumerate(VOCAB)}
idx2char = {i: c for c, i in char2idx.items()}

# Convertidores
def word_to_indices(word, max_len):
    indices = [char2idx[c] for c in word]
    indices += [char2idx["<pad>"]] * (max_len - len(indices))
    return indices

def labels_from_syllables(word, syllabified):
    labels = []
    pos = 0
    parts = syllabified.split("|")
    for part in parts[:-1]:  # no corte después del último
        pos += len(part)
        labels.extend([0]*(len(part)-1) + [1])
    labels.extend([0]*(len(parts[-1])))  # última sílaba
    return labels

# Dataset PyTorch
class SyllableDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
        self.max_len = max(len(w) for w, _ in samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        word, syllabified = self.samples[idx]
        x = word_to_indices(word, self.max_len)
        y = labels_from_syllables(word, syllabified)
        y += [0] * (self.max_len - len(y))
        return torch.tensor(x), torch.tensor(y)

# Red neuronal mínima
class SyllableSegmenter(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        emb = self.embedding(x)                         # [B, T, D]
        lstm_out, _ = self.lstm(emb)                    # [B, T, 2H]
        logits = self.classifier(lstm_out).squeeze(-1)  # [B, T]
        return logits

# Función entrenamiento
def train(model, dataloader, epochs=500, lr=1e-2):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        total_loss = 0
        for x, y in dataloader:
            logits = model(x)
            loss = F.binary_cross_entropy_with_logits(logits, y.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

# Decodificador
def predict(model, word):
    model.eval()
    x = torch.tensor([word_to_indices(word, len(word))])
    with torch.no_grad():
        logits = model(x)
        probs = torch.sigmoid(logits).squeeze(0)
    result = ""
    for i, c in enumerate(word):
        result += c
        if i < len(probs) and probs[i] > 0.5:
            result += "|"
    return result

# === Entrenamiento ===


# ------------------------------------------
data = fue


data = SyllableDataset(dataset)
loader = DataLoader(data, batch_size=1, shuffle=True)

model = SyllableSegmenter(vocab_size=len(VOCAB))
train(model, loader)

# === Pruebas ===
test_words = ["flores", "frutas", "legumbres", "dulce", "murciélago", "gracias"]
for word in test_words:
    print(word, "→", predict(model, word))

In [ ]:
#------------------------------------------
fuera = [
    ("Manzana", "Man-za-na"),
    ("Librería", "Li-bre-rí-a"),
    ("Años", "A-ños"),
    ("Pingüino", "Pin-güi-no"),
    ("Aston", "As-ton"),
    ("Computadora", "Com-pu-ta-do-ra"),
    ("Algoritmo", "Al-go-rit-mo"),
    ("Perro", "Pe-rro"),
    ("Gato", "Ga-to"),
    ("Elefante", "E-le-fan-te"),
    ("Sol", "Sol"),
    ("Luna", "Lu-na"),
    ("Universidad", "U-ni-ver-si-dad"),
    ("Informática", "In-for-má-ti-ca"),
    ("Desarrollo", "De-sa-rro-llo"),
    ("Programación", "Pro-gra-ma-ción"),
    ("Inteligencia", "In-te-li-gen-cia"),
    ("Artificial", "Ar-ti-fi-cial"),
    ("Aprendizaje", "A-pren-di-za-je"),
    ("Máquina", "Má-qui-na"),
    ("Datos", "Da-tos"),
    ("Análisis", "A-ná-li-sis"),
    ("Estadística", "Es-ta-dís-ti-ca"),
    ("Modelo", "Mo-de-lo"),
    ("Redes", "Re-des"),
    ("Neuronales", "Neu-ro-na-les"),
    ("Profundo", "Pro-fun-do"),
    ("Visión", "Vi-sión"),
    ("Lenguaje", "Len-gua-je"),
    ("Natural", "Na-tu-ral"),
    ("Procesamiento", "Pro-ce-sa-mien-to"),
    ("Robótica", "Ro-bó-ti-ca"),
    ("Sistemas", "Sis-te-mas"),
    ("Operativos", "O-pe-ra-ti-vos"),
    ("Software", "Soft-ware"),
    ("Hardware", "Hard-ware"),
    ("Internet", "In-ter-net"),
    ("Nube", "Nu-be"),
    ("Seguridad", "Se-gu-ri-dad"),
    ("Ciberseguridad", "Ci-ber-se-gu-ri-dad"),
    ("Encriptación", "En-crip-ta-ción"),
    ("Blockchain", "Block-chain"), # Las palabras no españolas pueden ser un reto
    ("Criptomoneda", "Crip-to-mo-ne-da"),
    ("Física", "Fí-si-ca"),
    ("Química", "Quí-mi-ca"),
    ("Biología", "Bio-lo-gía"),
    ("Matemáticas", "Ma-te-má-ti-cas"),
    ("Historia", "His-to-ria"),
    ("Geografía", "Ge-o-gra-fía"),
    ("Literatura", "Li-te-ra-tu-ra"),
    ("Filosofía", "Fi-lo-so-fí-a"),
    ("Psicología", "Psi-co-lo-gí-a"),
    ("Economía", "E-co-no-mí-a"),
    ("Política", "Po-lí-ti-ca"),
    ("Derecho", "De-re-cho"),
    ("Medicina", "Me-di-ci-na"),
    ("Ingeniería", "In-ge-nie-rí-a"),
    ("Arquitectura", "Ar-qui-tec-tu-ra"),
    ("Diseño", "Di-se-ño"),
    ("Arte", "Ar-te"),
    ("Música", "Mú-si-ca"),
    ("Deporte", "De-por-te"),
    ("Comida", "Co-mi-da"),
    ("Bebida", "Be-bi-da"),
    ("Viaje", "Via-je"),
    ("Vacaciones", "Va-ca-cio-nes"),
    ("Amigo", "A-mi-go"),
    ("Familia", "Fa-mi-lia"),
    ("Amor", "A-mor"),
    ("Felicidad", "Fe-li-ci-dad"),
    ("Tristeza", "Tris-te-za"),
    ("Enojo", "E-no-jo"),
    ("Sorpresa", "Sor-pre-sa"),
    ("Miedo", "Mie-do"),
    ("Esperanza", "Es-pe-ran-za"),
    ("Sueño", "Sue-ño"),
    ("Realidad", "Re-a-li-dad"),
    ("Conocimiento", "Co-no-ci-mien-to"),
    ("Sabiduría", "Sa-bi-du-rí-a"),
    ("Verdad", "Ver-dad"),
    ("Mentira", "Men-ti-ra"),
    ("Justicia", "Jus-ti-cia"),
    ("Libertad", "Li-ber-tad"),
    ("Paz", "Paz"),
    ("Guerra", "Gue-rra"),
    ("Tierra", "Tie-rra"),
    ("Agua", "A-gua"),
    ("Fuego", "Fue-go"),
    ("Aire", "Ai-re"),
    ("Naturaleza", "Na-tu-ra-le-za"),
    ("Animales", "A-ni-ma-les"),
    ("Plantas", "Plan-tas"),
    ("Flores", "Flo-res"),
    ("Árboles", "Ár-bo-les"),
    ("Montaña", "Mon-ta-ña"),
    ("Río", "Rí-o"),
    ("Mar", "Mar"),
    ("Océano", "O-cé-a-no"),
    ("Cielo", "Cie-lo"),
    ("Estrellas", "Es-tre-llas"),
    ("Planeta", "Pla-ne-ta"),
    ("Universo", "U-ni-ver-so"),
    ("Tiempo", "Tiem-po"),
    ("Espacio", "Es-pa-cio"),
    ("Energía", "E-ner-gí-a"),
    ("Materia", "Ma-te-ria"),
    ("Luz", "Luz"),
    ("Sonido", "So-ni-do"),
    ("Silencio", "Si-len-cio"),
    ("Color", "Co-lor"),
    ("Forma", "For-ma"),
    ("Tamaño", "Ta-ma-ño"),
    ("Peso", "Pe-so"),
    ("Velocidad", "Ve-lo-ci-dad"),
    ("Fuerza", "Fuer-za"),
    ("Movimiento", "Mo-vi-mien-to"),
    ("Reposo", "Re-po-so"),
    ("Origen", "O-ri-gen"),
    ("Destino", "Des-ti-no"),
    ("Principio", "Prin-ci-pio"),
    ("Fin", "Fin"),
    ("Causa", "Cau-sa"),
    ("Efecto", "E-fec-to"),
    ("Problema", "Pro-ble-ma"),
    ("Solución", "So-lu-ción"),
    ("Pregunta", "Pre-gun-ta"),
    ("Respuesta", "Res-pues-ta"),
    ("Idea", "I-de-a"),
    ("Pensamiento", "Pen-sa-mien-to"),
    ("Comunicación", "Co-mu-ni-ca-ción"),
    ("Información", "In-for-ma-ción"),
    ("Mensaje", "Men-sa-je"),
    ("Discurso", "Dis-cur-so"),
    ("Historia", "His-to-ria"),
    ("Relato", "Re-la-to"),
    ("Novela", "No-ve-la"),
    ("Poesía", "Po-e-sí-a"),
    ("Teatro", "Te-a-tro"),
    ("Cine", "Ci-ne"),
    ("Fotografía", "Fo-to-gra-fí-a"),
    ("Pintura", "Pin-tu-ra"),
    ("Escultura", "Es-cul-tu-ra"),
    ("Danza", "Dan-za"),
    ("Deporte", "De-por-te"),
    ("Juego", "Jue-go"),
    ("Trabajo", "Tra-ba-jo"),
    ("Dinero", "Di-ne-ro"),
    ("Riqueza", "Ri-que-za"),
    ("Pobreza", "Po-bre-za"),
    ("Éxito", "É-xi-to"),
    ("Fracaso", "Fra-ca-so"),
    ("Victoria", "Vic-to-ria"),
    ("Derrota", "De-rro-ta"),
    ("Riesgo", "Ries-go"),
    ("Oportunidad", "O-por-tu-ni-dad"),
    ("Desafío", "De-sa-fí-o"),
    ("Logro", "Lo-gro"),
    ("Esfuerzo", "Es-fuer-zo"),
    ("Dedicación", "De-di-ca-ción"),
    ("Disciplina", "Dis-ci-pli-na"),
    ("Constancia", "Cons-tan-cia"),
    ("Paciencia", "Pa-cien-cia"),
    ("Persistencia", "Per-sis-ten-cia"),
    ("Voluntad", "Vo-lun-tad"),
    ("Creación", "Cre-a-ción"),
    ("Destrucción", "Des-truc-ción"),
    ("Evolución", "E-vo-lu-ción"),
    ("Revolución", "Re-vo-lu-ción"),
    ("Transformación", "Trans-for-ma-ción"),
    ("Cambio", "Cam-bio"),
    ("Permanencia", "Per-ma-nen-cia"),
    ("Inicio", "I-ni-cio"),
    ("Final", "Fi-nal"),
    ("Pasado", "Pa-sa-do"),
    ("Presente", "Pre-sen-te"),
    ("Futuro", "Fu-tu-ro"),
    ("Memoria", "Me-mo-ria"),
    ("Olvido", "Ol-vi-do"),
    ("Recuerdo", "Re-cuer-do"),
    ("Sueño", "Sue-ño"),
    ("Realidad", "Re-a-li-dad"),
    ("Imaginación", "I-ma-gi-na-ción"),
    ("Fantasía", "Fan-ta-sí-a"),
    ("Creencia", "Cre-en-cia"),
    ("Fe", "Fe"),
    ("Esperanza", "Es-pe-ran-za"),
    ("Duda", "Du-da"),
    ("Certeza", "Cer-te-za"),
    ("Posibilidad", "Po-si-bi-li-dad"),
    ("Imposibilidad", "Im-po-si-bi-li-dad"),
    ("Necesidad", "Ne-ce-si-dad"),
    ("Contingencia", "Con-tin-gen-cia"),
    ("Accidente", "Ac-ci-den-te"),
    ("Esencia", "E-sen-cia"),
    ("Existencia", "Exis-ten-cia"),
    ("Fenómeno", "Fe-nó-me-no"),
    ("Nóumeno", "Nóu-me-no"),
    ("Ser", "Ser"),
    ("Nada", "Na-da"),
    ("Vacío", "Va-cí-o"),
    ("Plenitud", "Ple-ni-tud"),
    ("Totalidad", "To-ta-li-dad"),
    ("Parte", "Par-te"),
    ("Unidad", "U-ni-dad"),
    ("Pluralidad", "Plu-ra-li-dad"),
    ("Identidad", "I-den-ti-dad"),
    ("Diferencia", "Di-fe-ren-cia"),
    ("Semejanza", "Se-me-jan-za"),
    ("Oposición", "O-po-si-ción"),
    ("Contradicción", "Con-tra-dic-ción"),
    ("Síntesis", "Sín-te-sis"),
    ("Análisis", "A-ná-li-sis"),
    ("Inducción", "In-duc-ción"),
    ("Deducción", "De-duc-ción"),
    ("Abstracción", "Abs-trac-ción"),
    ("Concreto", "Con-cre-to"),
    ("General", "Ge-ne-ral"),
    ("Particular", "Par-ti-cu-lar"),
    ("Universal", "U-ni-ver-sal"),
    ("Singular", "Sin-gu-lar"),
    ("Individual", "In-di-vi-dual"),
    ("Colectivo", "Co-lec-ti-vo"),
    ("Público", "Pú-bli-co"),
    ("Privado", "Pri-va-do"),
    ("Social", "So-cial"),
    ("Cultural", "Cul-tu-ral"),
    ("Económico", "E-co-nó-mi-co"),
    ("Político", "Po-lí-ti-co"),
    ("Ético", "É-ti-co"),
    ("Estético", "Es-té-ti-co"),
    ("Religioso", "Re-li-gio-so"),
    ("Científico", "Cien-tí-fi-co"),
    ("Tecnológico", "Tec-no-ló-gi-co"),
    ("Artístico", "Ar-tís-ti-co"),
    ("Filosófico", "Fi-lo-só-fi-co"),
    ("Educativo", "E-du-ca-ti-vo"),
    ("Legal", "Le-gal"),
    ("Médico", "Mé-di-co"),
    ("Militar", "Mi-li-tar"),
    ("Deportivo", "De-por-ti-vo"),
    ("Culinario", "Cu-li-na-rio"),
    ("Viajero", "Via-je-ro"),
    ("Comunicativo", "Co-mu-ni-ca-ti-vo"),
    ("Interactivo", "In-te-rac-ti-vo"),
    ("Creativo", "Cre-a-ti-vo"),
    ("Productivo", "Pro-duc-ti-vo"),
    ("Eficiente", "E-fi-cien-te"),
    ("Eficaz", "E-fi-caz"),
    ("Flexible", "Fle-xi-ble"),
    ("Adaptable", "A-dap-ta-ble"),
    ("Innovador", "In-no-va-dor"),
    ("Sostenible", "Sos-te-ni-ble"),
    ("Responsable", "Res-pon-sa-ble"),
    ("Ético", "É-ti-co"),
    ("Moral", "Mo-ral"),
    ("Justo", "Jus-to"),
    ("Injusto", "In-jus-to"),
    ("Bueno", "Bue-no"),
    ("Malo", "Ma-lo"),
    ("Verdadero", "Ver-da-de-ro"),
    ("Falso", "Fal-so"),
    ("Bello", "Be-llo"),
    ("Feo", "Fe-o"),
    ("Simple", "Sim-ple"),
    ("Complejo", "Com-ple-jo"),
    ("Fácil", "Fá-cil"),
    ("Difícil", "Di-fí-cil"),
    ("Grande", "Gran-de"),
    ("Pequeño", "Pe-que-ño"),
    ("Largo", "Lar-go"),
    ("Corto", "Cor-to"),
    ("Alto", "Al-to"),
    ("Bajo", "Ba-jo"),
    ("Ancho", "An-cho"),
    ("Estrecho", "Es-tre-cho"),
    ("Pesado", "Pe-sa-do"),
    ("Ligero", "Li-ge-ro"),
    ("Caliente", "Ca-lien-te"),
    ("Frío", "Frí-o"),
    ("Nuevo", "Nue-vo"),
    ("Viejo", "Vie-jo"),
    ("Joven", "Jo-ven"),
    ("Adulto", "A-dul-to"),
    ("Niño", "Ni-ño"),
    ("Mujer", "Mu-jer"),
    ("Hombre", "Hom-bre"),
    ("Persona", "Per-so-na"),
    ("Humano", "Hu-ma-no"),
    ("Vida", "Vi-da"),
    ("Muerte", "Muer-te"),
    ("Salud", "Sa-lud"),
    ("Enfermedad", "En-fer-me-dad"),
    ("Cuerpo", "Cuer-po"),
    ("Mente", "Men-te"),
    ("Espíritu", "Es-pí-ri-tu"),
    ("Alma", "Al-ma"),
    ("Corazón", "Co-ra-zón"),
    ("Fuerza", "Fuer-za"),
    ("Debilidad", "De-bi-li-dad"),
    ("Alegría", "A-le-grí-a"),
    ("Tristeza", "Tris-te-za"),
    ("Amargura", "A-mar-gu-ra"),
    ("Gozo", "Go-zo"),
    ("Pena", "Pe-na"),
    ("Dicha", "Di-cha"),
    ("Desdicha", "Des-di-cha"),
    ("Risa", "Ri-sa"),
    ("Llanto", "Llan-to"),
    ("Sonrisa", "Son-ri-sa"),
    ("Lágrima", "Lá-gri-ma"),
    ("murciélago", ["mur", "cié", "la", "go"]),
    ("dulce", ["dul", "ce"]),
    ("gracias", ["gra", "cias"]),
    ("Tierra", ["tie", "rra"]),
]

In [ ]:
fue = fuera
def silabas_a_vector(palabra, silabeo):
    palabra = palabra.replace(" ", "")
    silabas = silabeo.split("-")
    vector = [0] * (len(palabra) - 1)
    index = 0
    for silaba in silabas[:-1]:  # no ponemos corte tras la última sílaba
        index += len(silaba)
        if index < len(palabra):
            vector[index - 1] = 1
    return (palabra, vector)

# Generamos el nuevo dataset
dataset_vectorizado = [silabas_a_vector(pal, sil) for pal, sil in fue]

# Ejemplo de los primeros 5
for item in dataset_vectorizado[:5]:
    print(item)

## Tokenizador entrenado con RL

In [ ]:
# Nota: medido en la competencia de tokenizadores como rl_cuaderno y rl_arreglado
# (laboratorio/banco/tokenizador_rl.py).
import random # Para simular decisiones del agente inicialmente

# --- SIMULACIÓN DEL ENTORNO ---
class TextTokenizationEnvironment:
    def __init__(self, text, model_to_evaluate=None):
        self.text = text
        self.current_idx = 0
        self.current_token_chars = [] # Caracteres del token que se está construyendo
        self.final_tokens = []
        self.model_to_evaluate = model_to_evaluate # Un modelo de PLN simulado para evaluar la recompensa

    def reset(self):
        self.current_idx = 0
        self.current_token_chars = []
        self.final_tokens = []
        return self._get_state()

    def _get_state(self):
        # El estado debería ser una representación numérica. Aquí, simplificado para conceptualizar.
        remaining_text = self.text[self.current_idx:]
        current_token_str = "".join(self.current_token_chars)
        # En un sistema real, esto serían embeddings o features numéricas
        return {
            "remaining_text_len": len(remaining_text),
            "current_char": remaining_text[0] if remaining_text else None,
            "current_token_str": current_token_str,
            "num_final_tokens": len(self.final_tokens)
        }

    def step(self, action):
        reward = 0
        done = False
        next_char = self.text[self.current_idx] if self.current_idx < len(self.text) else None

        if action == "ADD_CHAR_TO_CURRENT_TOKEN":
            if next_char:
                self.current_token_chars.append(next_char)
                self.current_idx += 1
                # Pequeña recompensa negativa para preferir tokens más largos,
                # o no dar recompensa hasta el final.
                # Aquí, no damos recompensa inmediata, se evalúa al final.
            else:
                # No hay más caracteres para añadir
                done = True # O penalizar acción inválida
        elif action == "FINISH_TOKEN_AND_START_NEW":
            if self.current_token_chars:
                self.final_tokens.append("".join(self.current_token_chars))
                self.current_token_chars = []
                # Recompensa negativa por cada token creado para fomentar la compresión
                reward -= 0.1 # Pequeña penalización por cada token.
            else:
                # Intentar finalizar un token vacío, acción inválida
                reward -= 1.0 # Penalización mayor

        # Comprobar si hemos terminado de procesar todo el texto
        if self.current_idx >= len(self.text) and not self.current_token_chars:
            done = True
            # Calcular la recompensa final basada en la calidad de la tokenización
            final_tokenized_text = self.final_tokens
            if self.model_to_evaluate:
                # Aquí, se llamaría a tu modelo de PLN para evaluar el rendimiento
                # Por ejemplo, model_to_evaluate.get_performance(final_tokenized_text)
                # Simulamos una recompensa basada en la longitud y una "calidad" inventada
                simulated_quality = random.uniform(0.5, 1.0) # Simula que el modelo "funciona"
                reward += simulated_quality * 10 # Recompensa por la calidad del modelo
                reward -= len(final_tokenized_text) * 0.5 # Penalizar muchos tokens

            # Si hay un token a medio construir al final, lo añadimos
            if self.current_token_chars:
                self.final_tokens.append("".join(self.current_token_chars))
                self.current_token_chars = []
                reward -= 0.1 # Penalización por token adicional

        next_state = self._get_state()
        return next_state, reward, done

# --- AGENTE DE REINFORCEMENT LEARNING (Simplificado) ---
class RLAgent:
    def __init__(self):
        self.q_table = {} # Tabla Q para aprendizaje por refuerzo simple (para demostración)
        self.actions = ["ADD_CHAR_TO_CURRENT_TOKEN", "FINISH_TOKEN_AND_START_NEW"]
        self.epsilon = 0.9 # Para exploración vs. explotación (al principio explora mucho)

    def choose_action(self, state):
        # Convertir el estado a una clave de tabla Q (simplificado)
        state_key = (state["current_char"], state["current_token_str"])

        if random.uniform(0, 1) < self.epsilon:
            return random.choice(self.actions) # Exploración
        else:
            # Explotación (elegir la mejor acción conocida)
            if state_key in self.q_table:
                # Elegir la acción con el mayor valor Q
                return max(self.q_table[state_key], key=self.q_table[state_key].get)
            else:
                return random.choice(self.actions) # Si no hay datos, elige al azar

    def learn(self, state, action, reward, next_state):
        # Actualización de la tabla Q (algoritmo Q-learning básico)
        state_key = (state["current_char"], state["current_token_str"])
        next_state_key = (next_state["current_char"], next_state["current_token_str"])

        if state_key not in self.q_table:
            self.q_table[state_key] = {a: 0.0 for a in self.actions}

        # Simular max Q para el siguiente estado (si existe)
        max_q_next = 0.0
        if next_state_key in self.q_table:
            max_q_next = max(self.q_table[next_state_key].values())

        alpha = 0.1 # Tasa de aprendizaje
        gamma = 0.9 # Factor de descuento

        # Fórmula Q-learning: Q(s,a) = Q(s,a) + alpha * (reward + gamma * max_Q(s',a') - Q(s,a))
        self.q_table[state_key][action] += alpha * (reward + gamma * max_q_next - self.q_table[state_key][action])

# --- PROCESO DE ENTRENAMIENTO ---
def train_rl_tokenizer(text_data, num_episodes=1000):
    agent = RLAgent()
    env = TextTokenizationEnvironment(text_data, model_to_evaluate="dummy_model") # dummy_model simula tu modelo real

    for episode in range(num_episodes):
        state = env.reset()
        done = False
        total_reward = 0

        while not done:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.learn(state, action, reward, next_state)
            state = next_state
            total_reward += reward

        # Reducir epsilon con el tiempo para que el agente explore menos y explote más
        agent.epsilon = max(0.01, agent.epsilon * 0.995)

        if episode % 100 == 0:
            print(f"Episodio {episode}, Recompensa Total: {total_reward}, Epsilon: {agent.epsilon:.2f}")

    # Después del entrenamiento, podemos usar el agente para tokenizar
    print("\n--- Tokenización después del entrenamiento ---")
    env_eval = TextTokenizationEnvironment(text_data) # Nuevo entorno para evaluar sin aprendizaje
    state = env_eval.reset()
    done = False
    while not done:
        action = agent.choose_action(state) # Ahora el agente elige la mejor acción aprendida
        state, _, done = env_eval.step(action)
    print(f"Texto original: '{text_data}'")
    print(f"Tokens generados: {env_eval.final_tokens}")


# --- EJEMPLO DE USO ---
text_to_tokenize = "El mono come bananas"
train_rl_tokenizer(text_to_tokenize, num_episodes=5000)

## Difusión de texto

In [ ]:
pip install torch transformers tqdm

In [ ]:
# Nota: TextusDiffusioRestituo era este mismo modelo con comentarios; se unió aquí.
# Medido en el banco: laboratorio/banco/difusion_texto.py (resultados en laboratorio/banco/resultados-01-generativos.md).
# Correr en Google Colab
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
from tqdm import tqdm
from transformers import AutoTokenizer
from huggingface_hub import notebook_login
import torch.optim as optim # Importar optimizador
import torch.cuda.amp as amp # Importar para mixed precision

# Login to Hugging Face Hub
try:
    notebook_login()
except Exception as e:
    print(f"Error durante el login: {e}")
    print("Asegúrate de tener una cuenta en Hugging Face y un token con permisos de lectura.")

corpus = [
    "Había una vez un pequeño dragón que vivía en la cima de una montaña.",
    "El zorro engañó al cuervo para que cantara y dejara caer el queso.",
    "El lobo sopló y sopló, pero no pudo derribar la casa de ladrillos.",
    "Una tortuga ganó una carrera contra una liebre confiada y rápida."
]

# --- Configuración ---
SEQ_LEN = 32
D_MODEL = 128
N_HEADS = 8
N_LAYERS = 4
BATCH_SIZE = 4
MAX_STEPS_TRAINING = 10000
DIFFUSION_STEPS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# --- Cargar Tokenizador Pre-entrenado ---
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

PAD_ID = tokenizer.pad_token_id
UNK_ID = tokenizer.unk_token_id
MASK_ID = tokenizer.mask_token_id
BOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id

if PAD_ID is None:
    raise ValueError("El tokenizador no tiene un PAD token.")
if MASK_ID is None:
    print("Advertencia: El tokenizador no tiene un MASK token. Usando UNK_ID como MASK_ID.")
    MASK_ID = UNK_ID
if BOS_ID is None:
    print("Advertencia: El tokenizador no tiene un BOS/CLS token. Usando UNK_ID como BOS_ID.")
    BOS_ID = UNK_ID
if EOS_ID is None:
    print("Advertencia: El tokenizador no tiene un EOS/SEP token. Usando UNK_ID como EOS_ID.")
    EOS_ID = UNK_ID

VOCAB_SIZE = tokenizer.vocab_size
print(f"Tamaño del vocabulario: {VOCAB_SIZE}")
print(f"ID del token PAD: {PAD_ID}, MASK: {MASK_ID}, BOS: {BOS_ID}, EOS: {EOS_ID}")

data = []
for sentence in corpus:
    encoded = tokenizer(sentence, max_length=SEQ_LEN, padding='max_length', truncation=True, return_tensors='pt')
    ids = encoded['input_ids'].squeeze(0).tolist()
    data.append(ids)

# --- Mecanismo de Ruido (Difusión) ---
def add_noise_to_sequence_embeddings(embeddings, t, total_diffusion_steps, noise_variance_start=0.0001, noise_variance_end=0.01):
    beta_t = noise_variance_start + t * (noise_variance_end - noise_variance_start) / (total_diffusion_steps - 1)
    beta_t_tensor = torch.tensor(beta_t, device=embeddings.device, dtype=embeddings.dtype)
    noise = torch.randn_like(embeddings) * beta_t_tensor.sqrt()
    noisy_embeddings = embeddings + noise
    return noisy_embeddings

def get_batch():
    batch_data = random.sample(data, BATCH_SIZE)
    original_sequences = torch.tensor(batch_data, dtype=torch.long, device=device)

    with torch.no_grad():
        original_embeddings = model.token_embedding(original_sequences)

    t = torch.randint(0, DIFFUSION_STEPS, (BATCH_SIZE,), dtype=torch.long, device=device)

    noisy_embeddings_batch = torch.empty_like(original_embeddings)
    for i in range(BATCH_SIZE):
        noisy_embeddings_batch[i] = add_noise_to_sequence_embeddings(
            original_embeddings[i], t[i].item(), DIFFUSION_STEPS
        )

    mask = (original_sequences == PAD_ID)

    return original_sequences, original_embeddings, noisy_embeddings_batch, t, mask

# --- Modelo Denoising (Basado en Transformer Encoder) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]

class ImprovedDiffusionTextModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, seq_len, diffusion_steps):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_embedding = PositionalEncoding(d_model, seq_len)
        self.time_embedding = nn.Embedding(diffusion_steps, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, n_layers)

        self.norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, noisy_input_embeddings, t, src_key_padding_mask=None):
        x = self.pos_embedding(noisy_input_embeddings)

        time_emb = self.time_embedding(t).unsqueeze(1)
        time_emb = time_emb.expand(-1, x.size(1), -1)
        x = x + time_emb

        output = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)

        output = self.norm(output)
        logits = self.fc_out(output)

        return logits

# --- Entrenamiento ---
model = ImprovedDiffusionTextModel(VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS, SEQ_LEN, DIFFUSION_STEPS).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

if device.type == 'cuda':
    scaler = amp.GradScaler()

print("\n--- Iniciando entrenamiento del mini-difusor ---")
for step in tqdm(range(MAX_STEPS_TRAINING), desc="Entrenamiento"):
    original_seq, original_embeddings, noisy_embeddings, t, mask = get_batch()

    optimizer.zero_grad()

    if device.type == 'cuda':
        with amp.autocast():
            predicted_logits = model(noisy_embeddings, t, src_key_padding_mask=mask)
            loss = criterion(predicted_logits.view(-1, VOCAB_SIZE), original_seq.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        predicted_logits = model(noisy_embeddings, t, src_key_padding_mask=mask)
        loss = criterion(predicted_logits.view(-1, VOCAB_SIZE), original_seq.view(-1))
        loss.backward()
        optimizer.step()

    if step % (MAX_STEPS_TRAINING // 10) == 0:
        tqdm.write(f"Paso {step}, Pérdida: {loss.item():.4f}")

print("--- Entrenamiento completado ---")

# --- Proceso de Muestreo (Generación) ---
def generate_text(model, starting_prompt=None, num_steps=DIFFUSION_STEPS):
    model.eval()

    if starting_prompt:
        encoded_prompt = tokenizer(starting_prompt.lower(), max_length=SEQ_LEN, padding='max_length', truncation=True, return_tensors='pt')
        current_sequence_indices = encoded_prompt['input_ids'].squeeze(0).to(device)
        current_sequence_embeddings = model.token_embedding(current_sequence_indices)

        prompt_token_ids = tokenizer.encode(starting_prompt.lower(), add_special_tokens=False)
        prompt_len = len(prompt_token_ids) + 2

        for i in range(prompt_len, SEQ_LEN):
            if current_sequence_indices[i] == PAD_ID or current_sequence_indices[i] == UNK_ID:
                 current_sequence_indices[i] = MASK_ID
                 current_sequence_embeddings[i] = model.token_embedding(torch.tensor(MASK_ID, device=device))
    else:
        current_sequence_indices = torch.full((SEQ_LEN,), MASK_ID, dtype=torch.long, device=device)
        current_sequence_embeddings = model.token_embedding(current_sequence_indices)

    print(f"\n--- Generando texto en {num_steps} pasos de desruido ---")
    progress_bar = tqdm(reversed(range(num_steps)), desc="Desruido")

    for t_step in progress_bar:
        t_tensor = torch.full((1,), t_step, dtype=torch.long, device=device)

        with torch.no_grad():
            predicted_logits = model(current_sequence_embeddings.unsqueeze(0), t_tensor).squeeze(0)

        predicted_indices = torch.argmax(predicted_logits, dim=-1)

        current_sequence_indices = predicted_indices
        current_sequence_embeddings = model.token_embedding(current_sequence_indices)

        if t_step % (num_steps // 5) == 0 or t_step == 0:
            current_text = tokenizer.decode(current_sequence_indices.tolist(), skip_special_tokens=True)
            progress_bar.write(f"  t={t_step}: {current_text}")

    final_indices = current_sequence_indices.tolist()
    final_text = tokenizer.decode(final_indices, skip_special_tokens=True)
    return final_text

# --- Ejemplos de Generación ---
print("\n--- Generación de texto (desde ruido aleatorio) ---")
generated_full_noise = generate_text(model, num_steps=DIFFUSION_STEPS)
print(f"\nResultado final (desde ruido): {generated_full_noise}")

print("\n--- Generación de texto (completando un prompt) ---")
generated_from_prompt = generate_text(model, starting_prompt="el zorro", num_steps=DIFFUSION_STEPS)
print(f"\nResultado final (desde prompt 'el zorro'): {generated_from_prompt}")

print("\n--- Otro ejemplo de prompt ---")
generated_from_prompt_2 = generate_text(model, starting_prompt="había una vez", num_steps=DIFFUSION_STEPS)
print(f"\nResultado final (desde prompt 'había una vez'): {generated_from_prompt_2}")

print("\n--- Tu prompt específico: 'Hola mundo como estas menciones todas las imágenes' ---")
user_specific_prompt = "Hola mundo como estas menciones todas las imágenes"
generated_user_prompt = generate_text(model, starting_prompt=user_specific_prompt, num_steps=DIFFUSION_STEPS)
print(f"\nResultado final (desde tu prompt): {generated_user_prompt}")

## Clasificador BERT con corrección GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import random
import Levenshtein

# --- Configuración BERT (se mantiene igual) ---
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')
for param in bert.parameters():
    param.requires_grad = False

class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Sequential(
            nn.Linear(bert.config.hidden_size, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

bert_model = BERTClassifier(bert)

def evaluar_con_BERT(oracion):
    if not oracion.strip():
        return 0.0
    inputs = tokenizer(oracion, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        salida = bert_model(**inputs)
    return salida.item()

# --- Vocabulario (se mantiene igual) ---
letras = list("abcdefghijklmnopqrstuvwxyzáéíóúñ ")
vocab = {l: i for i, l in enumerate(letras)}
inv_vocab = {i: l for l, i in vocab.items()}

# --- Módulo de Self-Attention (nuevo) ---
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Asegurarse de que embed_dim sea divisible por num_heads
        assert (
            self.head_dim * num_heads == embed_dim
        ), "embed_dim debe ser divisible por num_heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(num_heads * self.head_dim, embed_dim)

    def forward(self, x):
        # x tiene forma (num_nodes, embed_dim) o (batch_size, sequence_length, embed_dim)
        # Para tu caso, es (num_nodes, embed_dim)
        N, embed_dim = x.shape

        # Necesitamos simular una dimensión de "batch" si no la tenemos.
        # Si x es una secuencia de caracteres de una frase, N es la longitud de la secuencia.
        # batch_size = 1, sequence_length = N
        x = x.unsqueeze(0) # Añadir dimensión de batch: (1, N, embed_dim)

        batch_size, sequence_length, _ = x.shape

        # Dividir los embeddings en múltiples cabezas
        x = x.reshape(batch_size, sequence_length, self.num_heads, self.head_dim)

        # Aplicar las transformaciones lineales para Q, K, V
        queries = self.queries(x) # (batch_size, N, num_heads, head_dim)
        keys = self.keys(x)     # (batch_size, N, num_heads, head_dim)
        values = self.values(x) # (batch_size, N, num_heads, head_dim)

        # Transponer para multiplicar las cabezas de atención en paralelo
        # (batch_size, num_heads, N, head_dim)
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Multiplicación de matrices (Query * Key_transposed) para obtener las puntuaciones de atención
        # energy: (batch_size, num_heads, N, N)
        energy = torch.matmul(queries, keys.transpose(-1, -2))

        # Escalar las puntuaciones (para evitar softmax en valores muy grandes)
        scaled_energy = energy / (self.head_dim ** 0.5)

        # Aplicar softmax para obtener los pesos de atención
        attention = F.softmax(scaled_energy, dim=-1) # (batch_size, num_heads, N, N)

        # Multiplicar los pesos por los Values
        # out: (batch_size, num_heads, N, head_dim)
        out = torch.matmul(attention, values)

        # Concatenar las cabezas y pasar por la capa de salida final
        # (batch_size, N, num_heads, head_dim) -> (batch_size, N, embed_dim)
        out = out.transpose(1, 2).reshape(batch_size, sequence_length, embed_dim)

        out = self.fc_out(out)

        # Eliminar la dimensión de batch que añadimos si solo es 1 frase
        return out.squeeze(0) # (N, embed_dim)

# --- GNN con Self-Attention (modificado) ---
class CorreccionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, vocab_size, num_layers=3, num_attention_heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, in_channels)

        # Nuevo módulo de Self-Attention
        self.self_attention = SelfAttention(in_channels, num_attention_heads)

        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))

        self.decoder = nn.Linear(hidden_channels, vocab_size)

    def forward(self, x, edge_index):
        x = self.emb(x) # x: (num_nodes, in_channels)

        # Aplicar Self-Attention a los embeddings antes de las capas GCN
        # Esto permite que cada embedding de letra se "enriquezca" con el contexto de toda la secuencia
        x_att = self.self_attention(x)

        # Opcional: Sumar la salida de atención con los embeddings originales (Residual connection)
        # x = x + x_att
        # O simplemente usar la salida de atención como entrada a GCN
        x = x_att

        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return self.decoder(x)

# --- Funciones de Utilidad (se mantienen igual o con pequeñas mejoras) ---
def introducir_errores_en_frase(texto, max_errores_por_palabra=1, prob_error_espacio=0.1):
    palabras = texto.split(' ')
    frase_corrupta_final = []

    for palabra in palabras:
        if not palabra:
            frase_corrupta_final.append('')
            continue

        palabra_chars = list(palabra)
        num_errores = random.randint(0, max_errores_por_palabra)

        for _ in range(num_errores):
            if not palabra_chars and len(palabra) > 0:
                continue

            error_tipo = random.choice(['sustituir', 'insertar', 'eliminar', 'transponer'])

            if error_tipo == 'sustituir' and len(palabra_chars) > 0:
                idx = random.randint(0, len(palabra_chars) - 1)
                palabra_chars[idx] = random.choice(letras[:-1])
            elif error_tipo == 'insertar':
                idx = random.randint(0, len(palabra_chars))
                palabra_chars.insert(idx, random.choice(letras[:-1]))
            elif error_tipo == 'eliminar' and len(palabra_chars) > 0:
                idx = random.randint(0, len(palabra_chars) - 1)
                palabra_chars.pop(idx)
            elif error_tipo == 'transponer' and len(palabra_chars) >= 2:
                idx1 = random.randint(0, len(palabra_chars) - 2)
                idx2 = idx1 + 1
                palabra_chars[idx1], palabra_chars[idx2] = palabra_chars[idx2], palabra_chars[idx1]

        frase_corrupta_final.append("".join(palabra_chars))

    final_text = []
    for i, word in enumerate(frase_corrupta_final):
        final_text.append(word)
        if i < len(frase_corrupta_final) - 1:
            if random.random() < prob_error_espacio:
                final_text.append(random.choice(['  ', ' ', '']))
            else:
                final_text.append(' ')

    return "".join(final_text).strip()

def frase_a_grafo(frase):
    chars = list(frase.lower())
    node_ids = torch.tensor([vocab.get(c, vocab[' ']) for c in chars], dtype=torch.long)

    if len(chars) > 1:
        edge_index = torch.tensor([[i, i+1] for i in range(len(chars)-1)], dtype=torch.long).t()
        edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    return Data(x=node_ids, edge_index=edge_index), chars

def nodos_a_frase(pred_logits):
    pred_ids = pred_logits.argmax(dim=1)
    return ''.join(inv_vocab[i.item()] for i in pred_ids)

def calcular_recompensa_avanzada(frase_generada, frase_objetivo, bert_score_generada):
    max_len = max(len(frase_generada), len(frase_objetivo))
    if max_len == 0:
        levenshtein_norm = 0.0
    else:
        dist = Levenshtein.distance(frase_generada, frase_objetivo)
        levenshtein_norm = dist / max_len

    reward_levenshtein = (1 - levenshtein_norm)
    reward = reward_levenshtein + bert_score_generada

    return reward

# --- Entrenamiento RL con Self-Attention (modificado) ---
# Re-instanciar GNN con el módulo de Self-Attention
gnn = CorreccionGNN(in_channels=16, hidden_channels=32, vocab_size=len(vocab), num_layers=3, num_attention_heads=4)
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.0005)

def entrenar_rl_avanzado(frase_original_limpia, epochs=50000, num_generations=5):
    print(f"Iniciando entrenamiento para la frase: '{frase_original_limpia}'")

    for epoch in range(epochs):
        frase_corrupta = introducir_errores_en_frase(frase_original_limpia,
                                                      max_errores_por_palabra=random.randint(1, 3),
                                                      prob_error_espacio=0.2)

        if not frase_corrupta.strip() or len(frase_corrupta) < 3:
            continue

        data, original_chars_corrupta = frase_a_grafo(frase_corrupta)

        if data.x.numel() == 0:
            continue

        candidate_log_probs = []
        candidate_phrases = []
        candidate_rewards = []

        for _ in range(num_generations):
            logits = gnn(data.x, data.edge_index)

            probs = F.softmax(logits, dim=1)
            if torch.isnan(probs).any():
                 print(f"Epoch {epoch}, Generation {_}: NaN detectado en probabilidades, saltando esta generación.")
                 continue

            probs = torch.clamp(probs, min=1e-8)
            probs = probs / probs.sum(dim=1, keepdim=True)

            try:
                m = torch.distributions.Categorical(probs)
                sampled_ids = m.sample()
            except ValueError as e:
                 print(f"Epoch {epoch}, Generation {_}: Error en muestreo de Categorical: {e}. Saltando esta generación.")
                 continue

            generated_frase_list = [inv_vocab.get(i.item(), ' ') for i in sampled_ids]
            generated_frase = "".join(generated_frase_list)

            bert_score = evaluar_con_BERT(generated_frase)
            reward = calcular_recompensa_avanzada(generated_frase, frase_original_limpia, bert_score)

            candidate_log_probs.append(m.log_prob(sampled_ids).mean())
            candidate_phrases.append(generated_frase)
            candidate_rewards.append(reward)

        if not candidate_rewards:
             print(f"Epoch {epoch}: No se generaron candidatos válidos. Saltando esta época.")
             continue

        candidate_rewards_tensor = torch.tensor(candidate_rewards)
        best_reward_idx = torch.argmax(candidate_rewards_tensor)
        best_reward = candidate_rewards_tensor[best_reward_idx]
        best_log_prob = candidate_log_probs[best_reward_idx]
        best_generated_phrase = candidate_phrases[best_reward_idx]

        baseline = torch.mean(candidate_rewards_tensor)

        loss = - (best_reward - baseline) * best_log_prob

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 500 == 0:
            print(f"--- Época {epoch} ---")
            print(f"Frase Corrupta: '{frase_corrupta}'")
            print(f"Frase Generada (mejor): '{best_generated_phrase}'")
            print(f"Recompensa (mejor): {best_reward:.4f}")
            print(f"Baseline: {baseline:.4f}")
            print(f"Pérdida: {loss.item():.4f}")
            print("-" * 30)

        if best_reward > 1.9:
             print(f"Umbral de recompensa {1.9} alcanzado. Deteniendo el entrenamiento.")
             break

# --- Ejecución del entrenamiento ---
frase_limpia_ejemplo = "el veloz murcielago hindu comia feliz cardillo y kiwi la cigueña un queso exquisito"
entrenar_rl_avanzado(frase_limpia_ejemplo, epochs=100000, num_generations=10)

## Agente de atención para clasificar texto

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

# 1. Tokenizador básico
def tokenize(sentence):
    return sentence.lower().split()

# 2. Vocabulario y embedding
vocab = ["la", "casa", "es", "enorme", "mente", "grande", "pequeña", "oscura", "luminosa"]
word2idx = {word: idx for idx, word in enumerate(vocab)}
vocab_size = len(vocab)
embedding_dim = 768

# 3. Simulación de clase esperada por frase
data = [
    ("la casa es enorme mente grande", 1),  # clase 1
    ("la casa es pequeña y luminosa", 0),   # clase 0
    ("la casa es grande", 1),
    ("la casa es oscura", 0)
]

# 4. Embedding
embedding = nn.Embedding(vocab_size, embedding_dim)

# 5. Agente RL (decide atención binaria por palabra)
class AttentionAgent(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, 16)
        self.fc2 = nn.Linear(16, 1)  # output: logit de atención

    def forward(self, word_embeddings):
        x = F.relu(self.fc1(word_embeddings))  # (seq_len, 16)
        logits = self.fc2(x).squeeze(-1)       # (seq_len,)
        probs = torch.sigmoid(logits)          # Probabilidad de atender
        dist = torch.distributions.Bernoulli(probs)
        actions = dist.sample()
        log_probs = dist.log_prob(actions)
        return actions, log_probs

# 6. Clasificador final
class Classifier(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.fc = nn.Linear(embed_dim, 2)  # 2 clases

    def forward(self, selected_embeddings):
        if selected_embeddings.size(0) == 0:
            return torch.tensor([0.5, 0.5])  # atención nula → predicción aleatoria
        pooled = selected_embeddings.mean(dim=0)
        return F.softmax(self.fc(pooled), dim=0)

# 7. Entrenamiento con REINFORCE
agent = AttentionAgent(embedding_dim)
classifier = Classifier(embedding_dim)
optimizer = torch.optim.Adam(list(agent.parameters()), lr=0.01)

for epoch in range(500):
    total_reward = 0.1
    print(f"\nÉpoca {epoch+1}")

    for sentence, target_class in data:
        tokens = tokenize(sentence)
        indices = torch.tensor([word2idx.get(tok, 0) for tok in tokens])
        embeds = embedding(indices)

        # El agente decide atención por palabra
        actions, log_probs = agent(embeds)
        selected_embeds = embeds[actions.bool()]

        # Clasificación y reward
        output = classifier(selected_embeds)
        predicted = torch.argmax(output).item()
        reward = 1.0 if predicted == target_class else 0.0
        total_reward += reward

        # Mostrar atención
        attended_words = [tok for tok, a in zip(tokens, actions) if a == 1]
        print(f"Frase: {sentence}")
        print(f"Atiende: {attended_words} | Clase: {predicted} | Reward: {reward}")

        # REINFORCE update
        loss = -log_probs.sum() * reward  # maximizar log prob de buenas decisiones
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Reward total en época {epoch+1}: {total_reward}/{len(data)}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

# 1. Tokenizador
def tokenize(sentence):
    return sentence.lower().split()

# 2. Vocabulario base
vocab = [
    "quién", "es", "el", "presidente", "de", "francia",
    "cuál", "capital", "alemania", "parís", "berlín",
    "actual", "ministro", "nombre", "ciudad", "españa",
    "la", "y", "un", # Añadir palabras comunes que podrían aparecer en frases
]
word2idx = {word: idx for idx, word in enumerate(vocab)}
vocab_size = len(vocab)
embedding_dim = 16

# 3. Dataset realista de preguntas
data = [
    ("quién es el presidente de francia", "macron"),
    ("cuál es la capital de alemania", "berlín"),
    ("quién es el ministro actual de francia", "desconocido"),
    ("cuál es el nombre de la ciudad capital de españa", "madrid"),
]

# Simulador de respuestas correctas
answer2idx = {"macron": 0, "berlín": 1, "desconocido": 2, "madrid": 3}
idx2answer = {v: k for k, v in answer2idx.items()}
n_classes = len(answer2idx)

# 4. Embedding
embedding = nn.Embedding(vocab_size, embedding_dim)

# 5. Agente con múltiples cabezas de atención
class MultiHeadAttentionAgent(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 3)  # output logits for 3 attention levels: ignorar (0), medio (1), fuerte (2)
            ) for _ in range(n_heads)
        ])

    def forward(self, word_embeddings):
        actions_all = []
        log_probs_all = []

        for head in self.heads:
            logits = head(word_embeddings)  # (seq_len, 3)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            actions = dist.sample()  # sampled attention level (0, 1, or 2)
            log_probs = dist.log_prob(actions)
            actions_all.append(actions)
            log_probs_all.append(log_probs)

        # Promedio de atención de todas las cabezas
        # Esto es un valor promedio de 0 a 2 para cada palabra
        avg_attention = torch.stack(actions_all).float().mean(dim=0)  # (seq_len,)
        # Promedio de log_probs para el cálculo de la pérdida
        avg_log_probs = torch.stack(log_probs_all).mean(dim=0)

        return avg_attention, avg_log_probs

# 6. Clasificador que usa atención ponderada
class Classifier(nn.Module):
    def __init__(self, embed_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(embed_dim, n_classes)

    def forward(self, embeddings, attention_weights):
        # Si no hay palabras atendidas (o pesos muy bajos), evitar división por cero
        if attention_weights.sum() == 0:
            # Retorna una distribución uniforme si no hay atención significativa
            return F.softmax(torch.ones(n_classes) / n_classes, dim=0)

        # Ponderar los embeddings por los pesos de atención
        # embeddings: (seq_len, embed_dim)
        # attention_weights: (seq_len,)
        weighted_embeddings = embeddings * attention_weights.unsqueeze(-1) # (seq_len, embed_dim)

        # Sumar los embeddings ponderados para obtener una representación de la frase
        pooled = weighted_embeddings.sum(dim=0) # (embed_dim,)

        # Pasar por la capa final del clasificador
        return F.softmax(self.fc(pooled), dim=0)

# 7. Entrenamiento
agent = MultiHeadAttentionAgent(embedding_dim, n_heads=3)
classifier = Classifier(embedding_dim, n_classes) # El clasificador no se entrena con RL aquí
optimizer = torch.optim.Adam(list(agent.parameters()), lr=0.01)

total_points = 0
goal_points = 12 # Si sumamos 2 por respuesta correcta, 12 puntos = 6 respuestas correctas

rewards_epoch = []

print("--- Inicio del Entrenamiento ---")

for epoch in range(50): # Limitar a 50 épocas para una ejecución más rápida
    epoch_points = 0

    # Para mostrar la evolución, solo imprimimos en épocas específicas
    if epoch == 0 or epoch == 49: # Imprimir la primera y la última época
        print(f"\n===== Época {epoch+1} =====")

    for question, correct_answer in data:
        tokens = tokenize(question)
        indices = torch.tensor([word2idx.get(tok, 0) for tok in tokens], dtype=torch.long)
        embeds = embedding(indices)

        # El agente decide atención por palabra
        attention_values, log_probs = agent(embeds)  # valores: 0 (ignorar), 1 (medio), 2 (fuerte)
        normalized_attn = attention_values / 2.0  # Escalar a [0, 1] para el clasificador

        # Clasificación y reward
        output = classifier(embeds, normalized_attn)
        predicted_idx = torch.argmax(output).item()
        correct_idx = answer2idx[correct_answer]

        # Recompensa continua: 2.0 por correcto, 1.0 por parcial, -1.0 por incorrecto
        if predicted_idx == correct_idx:
            reward = 2.0
        # Una recompensa parcial si la predicción es "desconocido" y la respuesta correcta es "desconocido"
        # o si la predicción es una palabra que está contenida en la respuesta correcta (no aplica bien aquí)
        elif idx2answer[predicted_idx] == "desconocido" and correct_answer == "desconocido":
            reward = 1.0 # Recompensa parcial por identificar "desconocido"
        else:
            reward = -1.0  # castigo

        # REINFORCE update: maximizar log prob de buenas decisiones
        # La pérdida es el negativo de la recompensa por la suma de los log_probs
        # Queremos que el agente aumente la probabilidad de las acciones que llevaron a recompensas altas.
        loss = -log_probs.sum() * reward

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_points += reward

        # Mostrar atención y resultados solo en épocas específicas
        if epoch == 0 or epoch == 49:
            # Para visualizar, consideramos "atendido" si el valor de atención es >= 1.0
            attended_words = [tok for i, tok in enumerate(tokens) if attention_values[i] >= 1.0]
            print(f"  Frase: '{question}'")
            print(f"    Atención (valores): {[f'{t}:{v.item():.1f}' for t, v in zip(tokens, attention_values)]}")
            print(f"    Atiende (>=1.0): {attended_words}")
            print(f"    Predicción: {idx2answer[predicted_idx]} (Correcta: {correct_answer}) | Reward: {reward:.1f}")

    total_points += epoch_points
    rewards_epoch.append(epoch_points)

    if epoch == 0 or epoch == 49:
        print(f"  Reward total en época {epoch+1}: {epoch_points:.1f}/{len(data)*2.0} (máximo posible)") # Max reward per epoch is 2.0 * len(data)

    if total_points >= goal_points:
        print(f"\nMeta de {goal_points} puntos alcanzada. Deteniendo el entrenamiento.")
        break

print("\n--- Entrenamiento Finalizado ---")
print(f"Puntos totales acumulados: {total_points:.1f}")
print(f"Últimas 5 recompensas por época: {rewards_epoch[-5:]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics.pairwise import cosine_similarity

# =======================
# Datos de entrenamiento
# =======================
datos = [
    ("quién es el presidente de francia", "macron"),
    ("cuál es la capital de alemania", "berlín"),
    ("quién es el ministro actual de francia", "desconocido"),
    ("cuál es el nombre de la ciudad capital de españa", "madrid")
]
respuestas_posibles = ["macron", "berlín", "desconocido", "madrid"]

# =======================
# Word Embedding Simulado
# =======================
def generar_embedding(palabra):
    random.seed(hash(palabra) % (2**32))
    return np.random.rand(10)

# =======================
# Módulo de Atención Hebbiana
# =======================
class HebbianAttention(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.attn_weights = nn.Parameter(torch.randn(vocab_size))

    def forward(self, x):
        return torch.sigmoid(self.attn_weights[x])

# =======================
# Modelo Simple de Predicción
# =======================
class QAAgent(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.attention = HebbianAttention(self.vocab_size)
        self.classifier = nn.Linear(10, len(respuestas_posibles))  # Embedding reducido a 10
        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.parameters(), lr=0.01)

    def tokenize(self, frase):
        return [self.vocab.index(pal) for pal in frase.split()]

    def embed(self, frase):
        return torch.tensor([generar_embedding(p) for p in frase.split()], dtype=torch.float)

    def forward(self, frase):
        indices = self.tokenize(frase)
        attn_vals = self.attention(torch.tensor(indices))
        embeds = self.embed(frase)
        attended = (attn_vals.unsqueeze(1) * embeds).sum(dim=0)
        logits = self.classifier(attended)
        return logits, attn_vals.detach().numpy()

# =======================
# Métrica de Recompensa
# =======================
def calcular_reward(pred, verdad):
    if pred == verdad:
        return 2.0
    elif pred in verdad or verdad in pred:
        return 0.5
    sim = cosine_similarity([generar_embedding(pred)], [generar_embedding(verdad)])[0, 0]
    return 2.0 * (sim - 0.5)  # Normaliza entre -1 y +1

# =======================
# Entrenamiento
# =======================
def entrenar(agent, epocas=50):
    historial = []
    for ep in range(1, epocas+1):
        total_reward = 0
        print(f"\n===== Época {ep} =====")
        ep_rewards = []
        for frase, verdad in datos:
            agent.optimizer.zero_grad()
            logits, attn_vals = agent(frase)
            probs = torch.softmax(logits, dim=0)
            pred_idx = torch.argmax(probs).item()
            pred = respuestas_posibles[pred_idx]
            loss = agent.loss_fn(logits.unsqueeze(0), torch.tensor([respuestas_posibles.index(verdad)]))
            loss.backward()
            agent.optimizer.step()
            reward = calcular_reward(pred, verdad)
            total_reward += reward
            ep_rewards.append(reward)

            atendidas = [f"{p}:{round(v,1)}" for p, v in zip(frase.split(), attn_vals)]
            claves = [p for p, v in zip(frase.split(), attn_vals) if v >= 1.0]
            print(f"  Frase: '{frase}'")
            print(f"    Atención (valores): {atendidas}")
            print(f"    Atiende (>=1.0): {claves}")
            print(f"    Predicción: {pred} (Correcta: {verdad}) | Reward: {round(reward,1)}")
        historial.append((ep, total_reward, ep_rewards))
    return historial

# =======================
# Visualización
# =======================
def visualizar_historial(historial):
    epocas = [ep for ep, _, _ in historial]
    rewards = [r for _, r, _ in historial]
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 4))
    plt.plot(epocas, rewards, marker="o")
    plt.title("Reward total por época")
    plt.xlabel("Época")
    plt.ylabel("Reward total")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# =======================
# Entrenamiento y visualización
# =======================
vocab = sorted(set(" ".join(f for f, _ in datos).split()))
agent = QAAgent(vocab)
historial = entrenar(agent, epocas=50)
visualizar_historial(historial)

In [ ]:


class SelfAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        Q, K, V = self.q(x), self.k(x), self.v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))
        attn = self.softmax(scores)
        return torch.matmul(attn, V)

class ActorCritic(nn.Module):
    def __init__(self, dim, n_actions):
        super().__init__()
        self.attn = SelfAttention(dim)
        self.actor = nn.Linear(dim, n_actions)
        self.critic = nn.Linear(dim, 1)

    def forward(self, x):
        x = self.attn(x)
        return self.actor(x), self.critic(x)

class Clasificador(nn.Module):
    def __init__(self, dim, n_classes):
        super().__init__()
        self.attn = SelfAttention(dim)
        self.classifier = nn.Linear(dim, n_classes)

    def forward(self, x):
        x = self.attn(x)
        return self.classifier(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

# 1. Tokenizador
def tokenize(sentence):
    return sentence.lower().split()

# 2. Vocabulario base
vocab = [
    "quién", "es", "el", "presidente", "de", "francia",
    "cuál", "capital", "alemania", "parís", "berlín",
    "actual", "ministro", "nombre", "ciudad", "españa"
]
word2idx = {word: idx for idx, word in enumerate(vocab)}
vocab_size = len(vocab)
embedding_dim = 16

# 3. Dataset realista de preguntas
data = [
    ("quién es el presidente de francia", "macron"),
    ("cuál es la capital de alemania", "berlín"),
    ("quién es el ministro actual de francia", "desconocido"),
    ("cuál es el nombre de la ciudad capital de españa", "madrid"),
]

# Simulador de respuestas correctas
answer2idx = {"macron": 0, "berlín": 1, "desconocido": 2, "madrid": 3}
idx2answer = {v: k for k, v in answer2idx.items()}
n_classes = len(answer2idx)

# 4. Embedding
embedding = nn.Embedding(vocab_size, embedding_dim)

# 5. Agente con múltiples cabezas de atención
class MultiHeadAttentionAgent(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 3)  # Atención: ignorar (0), medio (1), fuerte (2)
            ) for _ in range(n_heads)
        ])

    def forward(self, word_embeddings):
        actions_all = []
        log_probs_all = []

        for head in self.heads:
            logits = head(word_embeddings)  # (seq_len, 3)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            actions = dist.sample()  # clase 0, 1 o 2
            log_probs = dist.log_prob(actions)
            actions_all.append(actions)
            log_probs_all.append(log_probs)

        # Promedio de atención de todas las cabezas
        avg_attention = torch.stack(actions_all).float().mean(dim=0)  # (seq_len,)
        return avg_attention, torch.stack(log_probs_all).mean(dim=0)

# 6. Clasificador que usa atención ponderada
class Classifier(nn.Module):
    def __init__(self, embed_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(embed_dim, n_classes)

    def forward(self, embeddings, attention_weights):
        if attention_weights.sum() == 0:
            return F.softmax(torch.ones(n_classes) / n_classes, dim=0)
        weighted = (embeddings.T * attention_weights).T
        pooled = weighted.sum(dim=0)
        return F.softmax(self.fc(pooled), dim=0)

# 7. Entrenamiento
agent = MultiHeadAttentionAgent(embedding_dim, n_heads=3)
classifier = Classifier(embedding_dim, n_classes)
optimizer = torch.optim.Adam(list(agent.parameters()), lr=0.01)

total_points = 0
goal_points = 12

rewards_epoch = []

for epoch in range(50):
    epoch_points = 0
    for question, correct_answer in data:
        tokens = tokenize(question)
        indices = torch.tensor([word2idx.get(tok, 0) for tok in tokens])
        embeds = embedding(indices)

        attention_values, log_probs = agent(embeds)  # valores: 0 a 2
        normalized_attn = attention_values / 2  # escala [0,1]

        output = classifier(embeds, normalized_attn)
        pred = torch.argmax(output).item()
        correct = answer2idx[correct_answer]

        # Recompensa continua
        if pred == correct:
            reward = 2.0
        elif idx2answer[pred] in correct_answer:
            reward = 1.0
        else:
            reward = -1.0  # castigo

        loss = -log_probs.sum() * reward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_points += reward

    total_points += epoch_points
    rewards_epoch.append(epoch_points)

    if total_points >= goal_points:
        break

total_points, rewards_epoch[-5:]  # Últimas recompensas para mostrar evolución

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from transformers import AutoTokenizer, AutoModel

config = {
    "embedding_dim": 16,
    "n_attention_heads": 3,
    "attention_hidden_dim": 32,
    "attention_levels": 3,
    "learning_rate": 0.01,
    "epochs": 50,
    "goal_points": 12,
    "bert_model_name": "bert-base-uncased",
}

def tokenize(sentence):
    return sentence.lower().split()

vocab = [
    "quién", "es", "el", "presidente", "de", "francia",
    "cuál", "capital", "alemania", "parís", "berlín",
    "actual", "ministro", "nombre", "ciudad", "españa",
    "la", "y", "un",
]
word2idx = {word: idx for idx, word in enumerate(vocab)}
vocab_size = len(vocab)

data = [
    ("quién es el presidente de francia", "macron"),
    ("cuál es la capital de alemania", "berlín"),
    ("quién es el ministro actual de francia", "desconocido"),
    ("cuál es el nombre de la ciudad capital de españa", "madrid"),
]

answer2idx = {"macron": 0, "berlín": 1, "desconocido": 2, "madrid": 3}
idx2answer = {v: k for k, v in answer2idx.items()}
n_classes = len(answer2idx)

embedding = nn.Embedding(vocab_size, config["embedding_dim"])

class SimpleSelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.scale_factor = embed_dim ** 0.5

    def forward(self, embeddings):
        Q = self.W_q(embeddings)
        K = self.W_k(embeddings)
        V = self.W_v(embeddings)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale_factor
        attention_weights = F.softmax(attention_scores, dim=-1)
        attended_output = torch.matmul(attention_weights, V)
        return attended_output

class MultiHeadAttentionAgent(nn.Module):
    def __init__(self, embed_dim, n_heads, attention_hidden_dim, attention_levels):
        super().__init__()
        self.self_attention_module = SimpleSelfAttention(embed_dim)
        self.n_heads = n_heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, attention_hidden_dim),
                nn.ReLU(),
                nn.Linear(attention_hidden_dim, attention_levels)
            ) for _ in range(n_heads)
        ])

    def forward(self, word_embeddings):
        contextualized_embeddings = self.self_attention_module(word_embeddings)
        actions_all = []
        log_probs_all = []
        for head in self.heads:
            logits = head(contextualized_embeddings)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            actions = dist.sample()
            log_probs = dist.log_prob(actions)
            actions_all.append(actions)
            log_probs_all.append(log_probs)
        avg_attention = torch.stack(actions_all).float().mean(dim=0)
        avg_log_probs = torch.stack(log_probs_all).mean(dim=0)
        return avg_attention, avg_log_probs

class Classifier(nn.Module):
    def __init__(self, embed_dim, n_classes):
        super().__init__()
        self.fc = nn.Linear(embed_dim, n_classes)

    def forward(self, embeddings, attention_weights):
        if attention_weights.sum() == 0:
            return F.softmax(torch.ones(n_classes) / n_classes, dim=0)
        weighted_embeddings = embeddings * attention_weights.unsqueeze(-1)
        pooled = weighted_embeddings.sum(dim=0)
        return F.softmax(self.fc(pooled), dim=0)

agent = MultiHeadAttentionAgent(
    config["embedding_dim"],
    config["n_attention_heads"],
    config["attention_hidden_dim"],
    config["attention_levels"]
)
classifier = Classifier(config["embedding_dim"], n_classes)
optimizer = torch.optim.Adam(list(agent.parameters()), lr=config["learning_rate"])

total_points = 0
rewards_epoch = []

for epoch in range(config["epochs"]):
    epoch_points = 0
    if epoch == 0 or epoch == config["epochs"] - 1:
        print(f"===== Época {epoch+1} =====")
    for question, correct_answer in data:
        tokens = tokenize(question)
        indices = torch.tensor([word2idx.get(tok, 0) for tok in tokens], dtype=torch.long)
        embeds = embedding(indices)
        attention_values, log_probs = agent(embeds)
        normalized_attn = attention_values / (config["attention_levels"] - 1.0)
        output = classifier(embeds, normalized_attn)
        predicted_idx = torch.argmax(output).item()
        correct_idx = answer2idx[correct_answer]
        reward = 0.0
        if predicted_idx == correct_idx:
            reward = 2.0
        elif idx2answer[predicted_idx] == "desconocido" and correct_answer == "desconocido":
            reward = 1.0
        else:
            reward = -1.0
        loss = -log_probs.sum() * reward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_points += reward
        if epoch == 0 or epoch == config["epochs"] - 1:
            attended_words = [tok for i, tok in enumerate(tokens) if attention_values[i] >= 1.0]
            print(f"  Frase: '{question}'")
            print(f"    Atención (valores): {[f'{t}:{v.item():.1f}' for t, v in zip(tokens, attention_values)]}")
            print(f"    Atiende (>=1.0): {attended_words}")
            print(f"    Predicción: {idx2answer[predicted_idx]} (Correcta: {correct_answer}) | Reward: {reward:.1f}")
    total_points += epoch_points
    rewards_epoch.append(epoch_points)
    if epoch == 0 or epoch == config["epochs"] - 1:
        print(f"  Reward total en época {epoch+1}: {epoch_points:.1f}/{len(data)*2.0}")
    if total_points >= config["goal_points"]:
        print(f"Meta de {config['goal_points']} puntos alcanzada. Deteniendo el entrenamiento.")
        break

print("--- Entrenamiento Finalizado ---")
print(f"Puntos totales acumulados: {total_points:.1f}")
print(f"Últimas 5 recompensas por época: {rewards_epoch[-5:]}")

In [ ]:
import torch
import torch.nn as nn
import re
from typing import List, Tuple
from collections import defaultdict # Necesario para build_vocab
import unicodedata # Para manejar tildes y diéresis en vocales

# --- 1. Simple Syllable Tokenizer ---
class SimpleSyllableTokenizer:
    def __init__(self):
        self.vowels = "aeiouAEIOUáéíóúÁÉÍÓÚüÜ" # Considera tildes y diéresis
        self.vocab = {}
        self.reverse_vocab = {}
        self.special_tokens = {"<pad>": 0, "<unk>": 1}
        # Dígrafos (grupos inseparables de consonantes en español para silabificación básica)
        self.DIGRAPHS = {'ch', 'll', 'rr'} # 'gu', 'qu' son más complejos con vocales

    def _is_vowel(self, char):
        return char.lower() in self.vowels # Usar .lower() para consistencia

    def _is_consonant(self, char):
        return char.lower() not in self.vowels and char.isalpha() # Es consonante si no es vocal y es alfabético

    def _is_valid_cluster_start(self, c1, c2):
        # Simplificación: algunos clusters válidos al inicio de sílaba.
        # No es exhaustivo, solo para dar una idea.
        if c1.lower() == 'p' and c2.lower() == 's': return True
        if c1.lower() in ['b', 'c', 'd', 'f', 'g', 'p', 't'] and c2.lower() in ['l', 'r']: return True
        return False

    def _syllabify_word_core(self, word: str) -> List[str]:
        """
        Lógica central de silabificación para palabras que no son excepciones.
        Implementa una heurística simple para dividir la palabra en sílabas,
        buscando patrones como CV, CVC, y manejando el flujo.
        """
        idx = 0
        current_word_syllables = []
        word_lower = word.lower() # Trabajar con la palabra en minúsculas

        while idx < len(word_lower):
            syl_start = idx

            # 1. Consumir consonantes iniciales (ej. 'h' en 'hola', 'tr' en 'tres')
            while idx < len(word_lower) and self._is_consonant(word_lower[idx]):
                idx += 1

            # 2. Consumir vocales (la parte CV o V inicial de la sílaba)
            vowel_group_start = idx
            while idx < len(word_lower) and self._is_vowel(word_lower[idx]):
                idx += 1

            # Si no avanzamos después de intentar consumir consonantes y vocales,
            # o si solo hay consonantes al final, tomamos el resto.
            if idx == syl_start:
                current_word_syllables.append(word[syl_start:])
                break

            syl_candidate_end = idx # El final actual de la sílaba candidata (después de V o CV)

            # Lógica para añadir consonantes finales o manejar grupos CC
            # V-C-V (ej. ca-sa, ho-la): 'a' es vocal, 's' es consonante, 'a' es vocal. Separar antes de 's'.
            if idx < len(word_lower) and self._is_consonant(word_lower[idx]): # Hay una consonante después de las vocales
                # Caso C-V (la consonante va con la siguiente sílaba)
                if idx + 1 < len(word_lower) and self._is_vowel(word_lower[idx+1]):
                    # Check for digraphs like 'ch', 'll', 'rr' that stay together
                    if idx + 2 <= len(word_lower) and word_lower[idx:idx+2] in self.DIGRAPHS:
                        syl_candidate_end = idx + 2 # Consume the digraph
                    else:
                        # Simple C-V split: current syllable ends just before this consonant
                        pass # syl_candidate_end remains at 'idx' (end of vowel group)
                # Caso C-C (dos consonantes seguidas)
                elif idx + 1 < len(word_lower) and self._is_consonant(word_lower[idx+1]):
                    # Si no es un cluster válido al inicio de la *siguiente* sílaba, la primera C se va con la sílaba actual
                    # Ejemplo: 'mun-do', 'her-mo-so'. 'n' con 'mu', 'r' con 'he'.
                    if not self._is_valid_cluster_start(word_lower[idx], word_lower[idx+1]):
                        syl_candidate_end = idx + 1 # Incluye la primera consonante
                    else:
                        syl_candidate_end = idx + 2 # Incluye ambas consonantes (e.g., 'bl' en 'ha-blar')
                # Caso C al final de la palabra (ej. "sol") o antes de puntuación
                else:
                    syl_candidate_end = idx + 1 # Incluye la consonante simple

            # Añadir la sílaba capturada
            syllables_part = word[syl_start:syl_candidate_end]
            if syllables_part: # Asegurarse de no añadir sílabas vacías
                current_word_syllables.append(syllables_part)

            idx = syl_candidate_end # Mover el índice para la próxima sílaba

        return [s for s in current_word_syllables if s] or [word] # Asegurarse de que al menos la palabra original esté si no se dividió

    def tokenize(self, text: str) -> List[str]:
        """
        Tokeniza un texto completo en una lista de sílabas, aplicando las reglas de excepción.
        """
        text = text.lower().strip()
        # Dividir por espacios; re.findall captura palabras y signos de puntuación/símbolos por separado.
        raw_words = re.findall(r'\b\w+\b|[^\w\s]', text)

        all_syllables = []
        for word in raw_words:
            if not word.strip():
                continue

            # Normalizar para reglas de longitud (eliminar tildes para contar caracteres base)
            word_norm_for_len = unicodedata.normalize('NFKD', word).encode('ascii', 'ignore').decode('utf-8')

            # Reglas 1, 2 y 3: Palabras de 1, 2 o 3 caracteres NO se separan.
            # Esto cubre "sol", "la", "el", "los" (si su longitud normalizada es 3).
            if len(word_norm_for_len) <= 3:
                all_syllables.append(word)
                continue

            # Para palabras más largas que no son excepciones, aplicar la lógica de silabificación central.
            # ¡CORRECCIÓN AQUÍ! Llama a _syllabify_word_core
            all_syllables.extend(self._syllabify_word_core(word))

        return all_syllables

    def build_vocab(self, corpus: List[str], min_freq: int = 1):
        """
        Construye un vocabulario de sílabas a partir de un corpus dado.
        """
        syllable_counts = defaultdict(int)
        for text_doc in corpus:
            syllables = self.tokenize(text_doc)
            for syllable in syllables:
                syllable_counts[syllable] += 1

        self.vocab = self.special_tokens.copy()
        current_idx = len(self.special_tokens)

        # Ordenar para asegurar una asignación consistente de IDs
        for syllable in sorted(syllable_counts.keys()):
            if syllable_counts[syllable] >= min_freq and syllable not in self.vocab:
                self.vocab[syllable] = current_idx
                current_idx += 1
        self.reverse_vocab = {idx: syl for syl, idx in self.vocab.items()}

    def convert_syllables_to_ids(self, syllables: List[str]) -> List[int]:
        """Convierte una lista de sílabas en sus IDs numéricos."""
        return [self.vocab.get(s, self.vocab["<unk>"]) for s in syllables]

    def convert_ids_to_syllables(self, ids: List[int]) -> List[str]:
        """Convierte una lista de IDs numéricos de vuelta a sílabas."""
        return [self.reverse_vocab.get(i, "<unk>") for i in ids]

# --- 2. Syllable Input Embedding ---
# Capa para generar los embeddings vectoriales de las sílabas.

class SyllableInputEmbedding(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, pad_idx: int):
        super().__init__()
        # nn.Embedding crea una tabla de embeddings (matriz) donde cada fila es el vector de una sílaba.
        # `padding_idx` asegura que el embedding del token de padding sea un vector de ceros y no se actualice.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dim = embedding_dim

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        input_ids: Tensor de IDs de sílabas, shape (batch_size, seq_len)
        Returns: Tensor de embeddings, shape (batch_size, seq_len, embedding_dim)
        """
        embedded = self.embedding(input_ids)
        return embedded

# --- 3. Hierarchical Positional Embedding ---
# Capa para generar y añadir información posicional jerárquica.

class HierarchicalPositionalEmbedding(nn.Module):
    def __init__(self, embedding_dim: int, max_global_positions: int = 512):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.max_global_positions = max_global_positions # Máximo número de grupos de sílabas esperado

        # Embedding para la posición global del grupo (ej. para el 1er par, 2do par, etc.)
        # +1 porque los índices de posición generalmente empiezan en 1 (o 0 si se incluye padding).
        # max_global_positions + 1 permite índices de 0 a max_global_positions.
        self.global_pos_embedding = nn.Embedding(max_global_positions + 1, embedding_dim)

        # Embedding para la posición interna dentro del grupo (ej. 1 para la 1ra sílaba del par, 2 para la 2da)
        # Usamos tamaño 3 para direccionar índices 1 y 2 (el índice 0 no se usa o podría ser para padding implícito si se desea).
        self.internal_pos_embedding = nn.Embedding(3, embedding_dim)

    def forward(self, num_syllables_in_sequence: int, device: torch.device) -> torch.Tensor:
        """
        Calcula los embeddings posicionales jerárquicos para una única secuencia.
        num_syllables_in_sequence: Longitud (L) de la secuencia de sílabas.
        device: Dispositivo (CPU/GPU) donde se crearán los tensores.
        Returns: Tensor de embeddings posicionales, shape (L, embedding_dim)
        """
        if num_syllables_in_sequence == 0:
            return torch.empty(0, self.embedding_dim, device=device)

        global_positions_indices = []
        internal_positions_indices = []

        current_global_pos_val = 1 # Los grupos empiezan desde la posición 1
        for i in range(num_syllables_in_sequence):
            # Posición global del grupo actual, truncada si excede el máximo configurado
            actual_g_pos_idx = min(current_global_pos_val, self.max_global_positions)
            global_positions_indices.append(actual_g_pos_idx)

            # Posición interna: 1 para la primera sílaba del par, 2 para la segunda
            if i % 2 == 0: # Si es la primera sílaba de un par (índice 0, 2, 4...)
                internal_positions_indices.append(1)
            else: # Si es la segunda sílaba de un par (índice 1, 3, 5...)
                internal_positions_indices.append(2)
                current_global_pos_val += 1 # Solo se incrementa la posición global *después* de un par completo

        # Convertir listas de índices a tensores de PyTorch
        global_pos_tensor = torch.tensor(global_positions_indices, dtype=torch.long, device=device)
        internal_pos_tensor = torch.tensor(internal_positions_indices, dtype=torch.long, device=device)

        # Obtener los embeddings de las tablas (clamp es una seguridad para evitar índices fuera de rango)
        global_embedded = self.global_pos_embedding(global_pos_tensor.clamp(min=0, max=self.max_global_positions))
        internal_embedded = self.internal_pos_embedding(internal_pos_tensor.clamp(min=0, max=2)) # Max index for internal is 2

        # Sumar los embeddings jerárquicos para combinarlos
        hierarchical_pe = global_embedded + internal_embedded
        return hierarchical_pe

    def forward_batch(self, batch_syllable_counts: List[int], device: torch.device) -> torch.Tensor:
        """
        Calcula los embeddings posicionales jerárquicos para un batch de secuencias.
        Aplica padding si las secuencias tienen diferentes longitudes.

        batch_syllable_counts: Lista con el número de sílabas para cada secuencia en el batch (ej. [7, 2, 5]).
        device: Dispositivo (CPU/GPU) donde se crearán los tensores.
        Returns: Tensor de embeddings posicionales, shape (batch_size, max_seq_len_in_batch, embedding_dim)
        """
        max_len = max(batch_syllable_counts) if batch_syllable_counts else 0
        if max_len == 0: # Si todas las secuencias son vacías o el batch está vacío
            return torch.empty(len(batch_syllable_counts), 0, self.embedding_dim, device=device)

        batch_pe_list = []
        for count in batch_syllable_counts:
            pe_single_sequence = self.forward(count, device) # Shape: (count, embedding_dim)
            if count < max_len:
                # Si la secuencia es más corta que la máxima, añadir padding de ceros
                padding_size = max_len - count
                padding = torch.zeros(padding_size, self.embedding_dim, device=device)
                pe_single_sequence = torch.cat([pe_single_sequence, padding], dim=0)
            batch_pe_list.append(pe_single_sequence)

        return torch.stack(batch_pe_list)

# --- Ejemplo de Uso ---
if __name__ == '__main__':
    # Determinar el dispositivo a usar
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Usando dispositivo: {device}")

    print("--- Demostración del Syllable Tokenizer ---")
    tokenizer = SimpleSyllableTokenizer()

    # Corpus de ejemplo para construir el vocabulario
    corpus = [
        "Hola mundo hermoso",
        "El sol brilla",
        "Tres tristes tigres",
        "La casa es azul",
        "Los pies en la arena",
        "paralelepipedo", # Palabra larga para prueba de silabificación
        "computadora"
    ]

    # Probar la tokenización de las palabras clave del ejemplo y otras
    print(f"'Hola': {tokenizer.tokenize('Hola')}")         # Esperado: ['ho', 'la']
    print(f"'mundo': {tokenizer.tokenize('mundo')}")       # Esperado: ['mun', 'do']
    print(f"'hermoso': {tokenizer.tokenize('hermoso')}")   # Esperado: ['her', 'mo', 'so']
    print(f"'sol': {tokenizer.tokenize('sol')}")           # Esperado: ['sol'] (regla de 3 caracteres)
    print(f"'los': {tokenizer.tokenize('los')}")           # Esperado: ['los'] (regla de 3 caracteres y 's' final)
    print(f"'casa': {tokenizer.tokenize('casa')}")         # Esperado: ['ca', 'sa']
    print(f"'paralelepipedo': {tokenizer.tokenize('paralelepipedo')}") # Esperado: ['pa', 'ra', 'le', 'le', 'pi', 'pe', 'do']
    print(f"'computadora': {tokenizer.tokenize('computadora')}") # Esperado: ['com', 'pu', 'ta', 'do', 'ra']

    test_sentence = "Hola mundo hermoso"
    syllables = tokenizer.tokenize(test_sentence)
    print(f"\nTexto de prueba: '{test_sentence}'")
    print(f"Sílabas tokenizadas: {syllables}")

    # Construir el vocabulario con el corpus
    tokenizer.build_vocab(corpus, min_freq=1)
    print(f"\nVocabulario construido (total {len(tokenizer.vocab)} sílabas únicas):")
    # print(tokenizer.vocab) # Descomentar para ver el vocabulario completo

    # Convertir sílabas a IDs y viceversa
    syllable_ids = tokenizer.convert_syllables_to_ids(syllables)
    print(f"IDs de sílabas para '{test_sentence}': {syllable_ids}")
    print(f"Sílabas desde IDs: {tokenizer.convert_ids_to_syllables(syllable_ids)}")

    print("\n--- Demostración del Syllable Input Embedding ---")
    embedding_dim = 128 # Dimensión del vector de embedding
    pad_idx = tokenizer.vocab["<pad>"]
    vocab_size = len(tokenizer.vocab)

    input_embedding_layer = SyllableInputEmbedding(vocab_size, embedding_dim, pad_idx).to(device)

    # Preparar un batch de IDs para la capa de embedding
    syllables_seq1 = tokenizer.tokenize("Hola mundo hermoso")
    syllable_ids_seq1 = tokenizer.convert_syllables_to_ids(syllables_seq1)

    syllables_seq2 = tokenizer.tokenize("El sol")
    syllable_ids_seq2 = tokenizer.convert_syllables_to_ids(syllables_seq2)

    syllables_seq3 = tokenizer.tokenize("computadora")
    syllable_ids_seq3 = tokenizer.convert_syllables_to_ids(syllables_seq3)

    # Padding para crear un batch de longitudes uniformes
    max_seq_len = max(len(syllable_ids_seq1), len(syllable_ids_seq2), len(syllable_ids_seq3))

    padded_ids_list = []
    for ids in [syllable_ids_seq1, syllable_ids_seq2, syllable_ids_seq3]:
        padded_ids_list.append(ids + [pad_idx] * (max_seq_len - len(ids)))

    batch_input_ids = torch.tensor(padded_ids_list, dtype=torch.long).to(device)
    print(f"Batch de IDs de sílabas (shape {batch_input_ids.shape}):\n{batch_input_ids}")

    syllable_embeddings = input_embedding_layer(batch_input_ids)
    print(f"Embeddings de sílabas (shape {syllable_embeddings.shape})")

    print("\n--- Demostración del Hierarchical Positional Embedding ---")
    # max_global_positions debe ser suficiente para la secuencia más larga de grupos
    # (max_seq_len_syllables / 2 + un poco de margen).
    hierarchical_pe_layer = HierarchicalPositionalEmbedding(embedding_dim, max_global_positions=max_seq_len // 2 + 10).to(device)

    # Generar PE para una única secuencia (la primera del batch)
    num_syllables_seq1 = len(syllable_ids_seq1)
    pe_seq1_direct = hierarchical_pe_layer.forward(num_syllables_seq1, device=device)
    print(f"PE Jerárquico para la primera secuencia (len {num_syllables_seq1}, shape {pe_seq1_direct.shape})")

    # Generar PE para el batch completo
    batch_counts = [len(syllable_ids_seq1), len(syllable_ids_seq2), len(syllable_ids_seq3)]
    batch_pe = hierarchical_pe_layer.forward_batch(batch_counts, device=device)
    print(f"PE Jerárquico para el batch completo (shape {batch_pe.shape})")

    print("\n--- Combinación Final de Embeddings (Ejemplo para la primera secuencia del batch) ---")
    # Para obtener los embeddings de entrada finales para el Transformer, sumarías:
    # Syllable Embedding + Hierarchical Positional Embedding

    # Tomar los embeddings de sílaba de la primera secuencia del batch
    # Es importante tomar la parte sin padding para la suma con PE
    original_syllable_embeddings_seq1 = syllable_embeddings[0, :num_syllables_seq1, :]

    # Tomar los PE correspondientes a la primera secuencia del batch_pe (ya tiene el tamaño correcto)
    pe_for_seq1_from_batch = batch_pe[0, :num_syllables_seq1, :]

    # Sumar para obtener el embedding final de la primera secuencia
    final_embeddings_seq1 = original_syllable_embeddings_seq1 + pe_for_seq1_from_batch
    print(f"Embeddings finales para la primera secuencia (shape {final_embeddings_seq1.shape})")

    print("\n--- Verificación detallada de posiciones para 'Hola mundo hermoso' ---")
    # Sílabas tokenizadas por este tokenizer para "Hola mundo hermoso": ['ho', 'la', 'mun', 'do', 'her', 'mo', 'so']
    # Longitud: 7 sílabas
    # Grupo 1: 'ho', 'la' -> Global 1, Interna 1 y 2
    # Grupo 2: 'mun', 'do' -> Global 2, Interna 1 y 2
    # Grupo 3: 'her', 'mo' -> Global 3, Interna 1 y 2
    # Grupo 4: 'so' -> Global 4, Interna 1 (manejo de sílaba impar al final)

    # Imprimir los índices generados por la capa para verificar
    global_indices_check = []
    internal_indices_check = []

    current_g = 1
    for i in range(len(syllables_seq1)):
        global_indices_check.append(min(current_g, hierarchical_pe_layer.max_global_positions))
        if i % 2 == 0:
            internal_indices_check.append(1)
        else:
            internal_indices_check.append(2)
            current_g += 1

    print(f"Sílabas: {syllables_seq1}")
    print(f"Posiciones Globales (generadas por PE): {global_indices_check}")
    print(f"Posiciones Internas (generadas por PE): {internal_indices_check}")

In [ ]:
import torch
import torch.nn as nn
import math
from typing import List, Tuple # Import List and Tuple here

class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, embedding_dim: int, max_seq_len: int = 512):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len

        # Crear una tabla de embeddings posicionales (PE) una sola vez
        # No son parámetros entrenables, por eso usamos register_buffer
        # register_buffer asegura que el tensor se mueva con el modelo (CPU/GPU)
        # pero no se incluya en los parámetros a optimizar.

        pe = torch.zeros(max_seq_len, embedding_dim)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embedding_dim, 2).float() * (-math.log(10000.0) / embedding_dim))

        # Aplicar las funciones seno y coseno
        # Para dimensiones pares (0, 2, 4...)
        pe[:, 0::2] = torch.sin(position * div_term)
        # Para dimensiones impares (1, 3, 5...)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)

    def forward(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """
        Devuelve los embeddings posicionales para una secuencia de longitud `seq_len`.

        Args:
            seq_len: La longitud de la secuencia para la cual se necesitan los PEs.
            device: El dispositivo (CPU/GPU) donde se cargarán los PEs.

        Returns:
            Tensor de embeddings posicionales, shape (seq_len, embedding_dim).
        """
        # Asegurarse de que `seq_len` no exceda `max_seq_len`
        # Si la secuencia es más larga, se truncaría (or could be generated dynamically,
        # but for simplicity, we truncate or recommend increasing max_seq_len).
        if seq_len > self.max_seq_len:
            print(f"Advertencia: `seq_len` ({seq_len}) excede `max_seq_len` ({self.max_seq_len}). Los PEs se truncarán.")
            return self.pe[:self.max_seq_len, :].to(device)

        return self.pe[:seq_len, :].to(device)

    def forward_batch(self, batch_syllable_counts: List[int], device: torch.device) -> torch.Tensor:
        """
        Calcula los embeddings posicionales sinusoidales para un batch of sequences.
        Applies padding if sequences have different lengths.

        batch_syllable_counts: Lista con el número de sílabas para cada secuencia en el batch (ej. [7, 2, 5]).
        device: Dispositivo (CPU/GPU) where the tensors will be created.
        Returns: Tensor de embeddings posicionales, shape (batch_size, max_seq_len_in_batch, embedding_dim)
        """
        max_len = max(batch_syllable_counts) if batch_syllable_counts else 0
        if max_len == 0:
            return torch.empty(len(batch_syllable_counts), 0, self.embedding_dim, device=device)

        batch_pe_list = []
        for count in batch_syllable_counts:
            # Obtener los PEs for the length of this sequence
            pe_single_sequence = self.forward(count, device) # Shape: (count, embedding_dim)

            if count < max_len:
                # Si la secuencia es más corta que la máxima, añadir padding de ceros
                padding_size = max_len - count
                padding = torch.zeros(padding_size, self.embedding_dim, device=device)
                pe_single_sequence = torch.cat([pe_single_sequence, padding], dim=0)
            batch_pe_list.append(pe_single_sequence)

        return torch.stack(batch_pe_list)

In [ ]:
# Use the variables defined in the previous cell
# --- Demostración del Sinusoidal Positional Embedding ---
print("\n--- Demostración del Sinusoidal Positional Embedding ---")
# max_seq_len debe ser suficiente para la secuencia más larga esperada.
# Aquí usamos la longitud máxima del batch de ejemplo, pero en un modelo real sería un valor fijo (ej. 512, 1024).
max_seq_len_for_pe = max_seq_len # Usamos el max_seq_len del batch actual para el ejemplo
sinusoidal_pe_layer = SinusoidalPositionalEmbedding(embedding_dim, max_seq_len=max_seq_len_for_pe).to(device)

# Generar PE para una única secuencia (la primera del batch)
num_syllables_seq1 = len(syllable_ids_seq1)
pe_seq1_direct_sinusoidal = sinusoidal_pe_layer.forward(num_syllables_seq1, device=device)
print(f"PE Sinusoidal para la primera secuencia (len {num_syllables_seq1}, shape {pe_seq1_direct_sinusoidal.shape})")

# Generar PE para el batch completo
batch_counts = [len(syllable_ids_seq1), len(syllable_ids_seq2), len(syllable_ids_seq3)]
batch_pe_sinusoidal = sinusoidal_pe_layer.forward_batch(batch_counts, device=device)
print(f"PE Sinusoidal para el batch completo (shape {batch_pe_sinusoidal.shape})")

print("\n--- Combinación Final de Embeddings (Ejemplo con Sinusoidal PE) ---")
# Syllable Embedding + Sinusoidal Positional Embedding

# Tomar los embeddings de sílaba de la primera secuencia del batch (sin padding)
original_syllable_embeddings_seq1 = syllable_embeddings[0, :num_syllables_seq1, :]

# Tomar los PE sinusoidales correspondientes a la primera secuencia del batch_pe_sinusoidal
pe_for_seq1_from_batch_sinusoidal = batch_pe_sinusoidal[0, :num_syllables_seq1, :]

# Sumar para obtener el embedding final de la primera secuencia
final_embeddings_seq1_sinusoidal = original_syllable_embeddings_seq1 + pe_for_seq1_from_batch_sinusoidal
print(f"Embeddings finales para la primera secuencia (con Sinusoidal PE) (shape {final_embeddings_seq1_sinusoidal.shape})")

print("\n--- Verificación detallada de posiciones para 'Hola mundo hermoso' (con Sinusoidal PE) ---")
# Aquí ya no tiene sentido imprimir "Posiciones Globales" o "Internas"
# porque el embedding sinusoidal codifica la posición de forma única en un solo vector.
print(f"Sílabas: {syllables_seq1}")
# Para ver la diferencia, podrías imprimir parte del primer PE sinusoidal generado:
# print(f"Primeros elementos del PE para 'ho': {pe_seq1_direct_sinusoidal[0, :5].tolist()}")
# print(f"Primeros elementos del PE para 'la': {pe_seq1_direct_sinusoidal[1, :5].tolist()}")

# Notarás que cada fila del tensor `pe_seq1_direct_sinusoidal` es un vector único que representa la posición.
# No hay una descomposición en "global" e "interna" aquí.